# VIBTARVE visualizations: legacy audit cohort

**VLM-Initiated Behavior Tree Auditing of Robot VLA Execution (VIBTARVE)** is the research name; the `VIBTRAVE` filename is a retained legacy spelling. See [the research overview](../docs/vibtarve.md) for authorship, read-only protocol and manuscript-reported findings. This legacy notebook is not the 464-episode btaudit2 study and must not be used to imply that its plots reproduce the finalized paper results.

This notebook focuses on episodes **2-21** only, matching the evaluation protocol where episode 1 is BT-building and is excluded from mission-engine comparison.

Analysis structure:
1. Section 1: Data overview, distribution, and mission-engine comparison trends.
2. Section 2: RQ1 deep-dive visualizations and tables.
3. Section 3: RQ2 deep-dive visualizations and tables.

Core data processing is routed through `scripts/notebook_qwen_compare_utils.py` so this report is consistent with prior audit notebooks.

In [ ]:
from pathlib import Path
import importlib
import json
import re
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts').is_dir() and (p / 'README.md').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Start Jupyter from the repository or its notebooks directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

LOGS_DIR = PROJECT_ROOT / 'data' / 'logs'
SCHEDULE_PATH = LOGS_DIR / 'human_eval_schedule_final.xlsx'

import scripts.notebook_qwen_compare_utils as qutils
importlib.reload(qutils)

build_model_compare_dataframe = qutils.build_model_compare_dataframe
summarize_model_compare = qutils.summarize_model_compare
plot_model_compare = qutils.plot_model_compare
build_rq1_tables = qutils.build_rq1_tables
build_rq2_tables = qutils.build_rq2_tables
plot_rq1_bundle = qutils.plot_rq1_bundle
plot_rq2_bundle = qutils.plot_rq2_bundle
build_episode_node_completion_dataframe = qutils.build_episode_node_completion_dataframe
build_episode_completion_tables = qutils.build_episode_completion_tables
build_episode_master_export_table = qutils.build_episode_master_export_table
build_all_episode_workbook_rows = qutils.build_all_episode_workbook_rows
scenario_label = qutils.scenario_label

In [ ]:
def parse_episode_idx(episode_dir: object) -> float:
    s = '' if pd.isna(episode_dir) else str(episode_dir)
    m = re.search(r'episode_(\d+)_of_(\d+)', s)
    if not m:
        return np.nan
    return float(m.group(1))


def _norm_col_local(name: object) -> str:
    return ''.join(ch if ch.isalnum() else '_' for ch in str(name).strip().lower()).strip('_')


def _find_col_local(columns: list[object], aliases: list[str]) -> object:
    norm_to_raw = {_norm_col_local(c): c for c in columns}
    for alias in aliases:
        key = _norm_col_local(alias)
        if key in norm_to_raw:
            return norm_to_raw[key]
    for alias in aliases:
        key = _norm_col_local(alias)
        for norm_name, raw in norm_to_raw.items():
            if key in norm_name:
                return raw
    return None


def _read_tailored_review_sheet_schema_aware(wb_path: Path) -> pd.DataFrame:
    try:
        xl = pd.ExcelFile(wb_path)
    except Exception:
        return pd.DataFrame()

    candidates: list[str] = []
    for sheet_name in xl.sheet_names:
        norm_name = _norm_col_local(sheet_name)
        if ('sheet4' in norm_name and 'missionengine' in norm_name) or ('mod1' in norm_name and 'missionengine' in norm_name):
            candidates.append(sheet_name)
    if len(xl.sheet_names) > 3 and xl.sheet_names[3] not in candidates:
        candidates.append(xl.sheet_names[3])

    for sheet_name in candidates:
        for hdr in [7, 8, 6, 0]:
            try:
                df = pd.read_excel(wb_path, sheet_name=sheet_name, header=hdr)
            except Exception:
                continue
            if df.empty:
                continue
            norms = {_norm_col_local(c) for c in df.columns}
            if {'status_eval_id', 'model_status'}.issubset(norms):
                return df

    return pd.DataFrame()


def _build_episode_meta(logs_dir: Path, episode_dirs: list[Path]) -> pd.DataFrame:
    meta_rows: list[dict[str, object]] = []
    for ep in episode_dirs:
        summary_path = ep / 'episode_summary.json'
        summary = {}
        try:
            with open(summary_path, 'r', encoding='utf-8') as f:
                summary = json.load(f)
        except Exception:
            summary = {}

        info = summary.get('info_summary', {}) if isinstance(summary, dict) else {}
        meta_rows.append(
            {
                'episode_dir': ep.name,
                'episode_id_meta': str(summary.get('episode_id') or ''),
                'scenario_id': str(summary.get('scenario_id') or ''),
                'sim_success': info.get('success') if isinstance(info, dict) else np.nan,
            }
        )
    return pd.DataFrame(meta_rows)


def build_tailored_compare_dataframe(
    logs_dir: Path,
    schedule_path: Path,
    only_schedule_complete: bool = True,
    model_name: str = 'qwen3.5_tailored',
) -> pd.DataFrame:
    ep_dirs = sorted([p for p in logs_dir.glob('*episode_*_of_*_*') if p.is_dir()])
    keep_names = qutils._read_schedule_complete_episode_names(schedule_path) if only_schedule_complete else None
    if keep_names:
        ep_dirs = [p for p in ep_dirs if p.name in keep_names]

    meta_df = _build_episode_meta(logs_dir, ep_dirs)

    rows: list[dict[str, object]] = []
    for ep in ep_dirs:
        wb = ep / 'human_rater_evaluation.xlsx'
        if not wb.exists():
            continue

        sheet_df = _read_tailored_review_sheet_schema_aware(wb)
        if sheet_df.empty:
            continue

        c_eval = _find_col_local(list(sheet_df.columns), ['status_eval_id', 'node_id'])
        c_model_status = _find_col_local(list(sheet_df.columns), ['model_status', 'qwen35_status', 'qwen3_5_status'])
        c_model_progress = _find_col_local(list(sheet_df.columns), ['model_progress', 'model_progress_score', 'qwen35_progress_score'])
        c_model_time = _find_col_local(list(sheet_df.columns), ['video_time_m:ss', 'machine_time_m:ss', 'model_video_time_m:ss', 'model_video_time_mss'])
        c_reviewer_agree = _find_col_local(list(sheet_df.columns), ['reviewer_agree_YN_partial', 'reviewer_agrees_machine_YN_partial'])
        c_reviewer_status = _find_col_local(list(sheet_df.columns), ['reviewer_status'])
        c_reviewer_time = _find_col_local(list(sheet_df.columns), ['reviewer_time_m:ss', 'reviewer_completion_time_m:ss'])
        c_reviewer_notes = _find_col_local(list(sheet_df.columns), ['reviewer_notes'])
        c_episode_id = _find_col_local(list(sheet_df.columns), ['episode_id'])
        c_condition = _find_col_local(list(sheet_df.columns), ['condition'])

        for _, row in sheet_df.iterrows():
            rows.append(
                {
                    'model_name': model_name,
                    'episode_dir': ep.name,
                    'episode_id': str(row.get(c_episode_id, '') if c_episode_id else ''),
                    'condition': str(row.get(c_condition, '') if c_condition else ''),
                    'status_eval_id': str(row.get(c_eval, '') if c_eval else ''),
                    'model_status': row.get(c_model_status, '') if c_model_status else '',
                    'model_status_norm': qutils._norm_status(row.get(c_model_status, '') if c_model_status else ''),
                    'model_progress': pd.to_numeric(row.get(c_model_progress, np.nan), errors='coerce') if c_model_progress else np.nan,
                    'model_video_time_mss': row.get(c_model_time, '') if c_model_time else '',
                    'model_video_time_s': qutils._to_seconds_mmss(row.get(c_model_time, np.nan)) if c_model_time else np.nan,
                    'reviewer_agree': qutils._normalize_agree(row.get(c_reviewer_agree, '') if c_reviewer_agree else ''),
                    'reviewer_status': row.get(c_reviewer_status, '') if c_reviewer_status else '',
                    'reviewer_video_time_mss': row.get(c_reviewer_time, '') if c_reviewer_time else '',
                    'reviewer_video_time_s': qutils._to_seconds_mmss(row.get(c_reviewer_time, np.nan)) if c_reviewer_time else np.nan,
                    'reviewer_notes': row.get(c_reviewer_notes, '') if c_reviewer_notes else '',
                }
            )

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    out = out.merge(meta_df, on='episode_dir', how='left')
    epid = out['episode_id'].fillna('').astype(str).str.strip()
    out.loc[epid.eq(''), 'episode_id'] = out.loc[epid.eq(''), 'episode_id_meta'].fillna('').astype(str)
    cond = out['condition'].fillna('').astype(str).str.strip()
    out['scenario_id'] = np.where(cond.eq(''), out['scenario_id'], cond)

    out['reviewer_status_clean'] = out['reviewer_status'].fillna('').astype(str).str.strip()
    out['timing_delta_s'] = out['model_video_time_s'] - out['reviewer_video_time_s']
    out['timing_abs_delta_s'] = out['timing_delta_s'].abs()
    out['timing_aligned_2s'] = (
        out['reviewer_status_clean'].eq('')
        & out['timing_abs_delta_s'].notna()
        & (out['timing_abs_delta_s'] <= 2.0)
    )
    out['agreement_effective'] = out['reviewer_agree'].fillna('')
    out.loc[out['agreement_effective'].eq('') & out['timing_aligned_2s'], 'agreement_effective'] = 'timing_aligned'
    return out


BASE_COMPARE_DF = build_model_compare_dataframe(
    logs_dir=LOGS_DIR,
    schedule_path=SCHEDULE_PATH,
    only_schedule_complete=True,
)

TAILORED_COMPARE_DF = build_tailored_compare_dataframe(
    logs_dir=LOGS_DIR,
    schedule_path=SCHEDULE_PATH,
    only_schedule_complete=True,
    model_name='qwen3.5_tailored',
)

if BASE_COMPARE_DF.empty and TAILORED_COMPARE_DF.empty:
    display(Markdown('### No rows found from workbook sheets. Check logs and schedule files.'))

BASE_PLUS_TAILORED_DF = pd.concat(
    [df for df in [BASE_COMPARE_DF, TAILORED_COMPARE_DF] if not df.empty],
    ignore_index=True,
    sort=False,
    )

BASE_PLUS_TAILORED_DF = BASE_PLUS_TAILORED_DF.copy()
BASE_PLUS_TAILORED_DF['episode_idx'] = BASE_PLUS_TAILORED_DF['episode_dir'].map(parse_episode_idx)
COMPARE_DF = BASE_PLUS_TAILORED_DF[BASE_PLUS_TAILORED_DF['episode_idx'].between(2, 21, inclusive='both')].copy()

COMPARE_DF['run_id'] = COMPARE_DF['episode_dir'].astype(str).str.extract(r'(_[0-9a-f]{8})$')[0].fillna('unknown')
COMPARE_DF['episode_key'] = COMPARE_DF['scenario_id'].astype(str) + '|' + COMPARE_DF['run_id'].astype(str) + '|' + COMPARE_DF['episode_idx'].astype('Int64').astype(str)

display(Markdown('## Loaded Comparison Dataset (Episodes 2-21 Only)'))
display(COMPARE_DF[['model_name', 'scenario_id', 'episode_dir', 'episode_idx', 'status_eval_id']].head(12))

model_rows = (
    COMPARE_DF.groupby('model_name', dropna=False)
    .agg(rows=('model_name', 'size'), episodes=('episode_key', 'nunique'))
    .reset_index()
    .sort_values('model_name')
    .reset_index(drop=True)
)
display(Markdown('### Model coverage (including tailored qwen3.5 run)'))
display(model_rows)

run_count = COMPARE_DF[['scenario_id', 'run_id']].drop_duplicates().shape[0]
scenario_count = COMPARE_DF['scenario_id'].nunique()
episode_keys = COMPARE_DF[['scenario_id', 'run_id', 'episode_idx']].drop_duplicates().shape[0]

display(Markdown(f'Observed scenarios: **{scenario_count}**  \nObserved scenario-run groups: **{run_count}**  \nObserved episode slots (2-21 only): **{episode_keys}** (target = 300)'))

In [ ]:
# Load standardized episode-workbook dataframe for visualization work.
EPISODE_WORKBOOK_DF = build_all_episode_workbook_rows(
    logs_dir=LOGS_DIR,
    max_nodes=4,
    export_csv=False,
)

if EPISODE_WORKBOOK_DF.empty:
    display(Markdown('### Episode workbook dataframe is empty.'))
else:
    EPISODE_WORKBOOK_DF = EPISODE_WORKBOOK_DF.copy()
    EPISODE_WORKBOOK_DF['episode_index'] = pd.to_numeric(EPISODE_WORKBOOK_DF['episode_index'], errors='coerce')

    # Convenience numeric encoding for plotting/modeling while preserving original boolean column.
    EPISODE_WORKBOOK_DF['success_int'] = EPISODE_WORKBOOK_DF['success'].map(
        lambda v: 1 if v is True else (0 if v is False else np.nan)
    )

    EPISODE_WORKBOOK_2_21 = EPISODE_WORKBOOK_DF[
        EPISODE_WORKBOOK_DF['episode_index'].between(2, 21, inclusive='both')
    ].copy()

    # Stable aliases for downstream visualization cells.
    VIZ_DF_ALL = EPISODE_WORKBOOK_DF
    VIZ_DF = EPISODE_WORKBOOK_2_21

    display(Markdown('## Episode Workbook Dataframe (Visualization-Ready)'))
    display(
        VIZ_DF[
            [
                'scenario_id',
                'episode_dir',
                'episode_index',
                'episode_id',
                'bt_action_nodes',
                'condition',
                'success',
                'success_int',
                'mission_name',
            ]
        ].head(12)
    )
    display(Markdown(
        f"All episodes shape: **{VIZ_DF_ALL.shape[0]} rows x {VIZ_DF_ALL.shape[1]} columns**  " +
        f"Episodes 2-21 shape: **{VIZ_DF.shape[0]} rows x {VIZ_DF.shape[1]} columns**"
    ))

In [ ]:
# Build TP/FP/TN/FN diagnostics from completion-time coverage and add columns for downstream use.

MODEL_SPECS = [
    {
        "key": "qwen3",
        "label": "qwen3",
        "completion_prefix": "model_completion_time",
    },
    {
        "key": "qwen35",
        "label": "qwen3.5",
        "completion_prefix": "qwen3_5_model_completion_time",
    },
    {
        "key": "qwen35_mod1",
        "label": "qwen3.5 mod1",
        "completion_prefix": "qwen3_5_mod1_completion_time",
    },
]


def _safe_int(v: object, default: int = 0) -> int:
    try:
        if pd.isna(v):
            return default
        return int(v)
    except Exception:
        return default


def _is_time_filled(v: object) -> bool:
    if pd.isna(v):
        return False
    return str(v).strip() != ""


def _pred_success_from_completion_times(row: pd.Series, prefix: str) -> tuple[int, int, bool]:
    required = max(0, _safe_int(row.get("bt_action_nodes", 0), default=0))
    if required == 0:
        return 0, 0, False

    filled = 0
    for i in range(1, required + 1):
        col = f"node_{i}_{prefix}"
        filled += int(_is_time_filled(row.get(col, np.nan)))

    return required, filled, bool(filled == required)


def _pred_success_from_reviewer_times(row: pd.Series) -> tuple[int, int, bool]:
    required = max(0, _safe_int(row.get("bt_action_nodes", 0), default=0))
    if required == 0:
        return 0, 0, False

    filled = 0
    for i in range(1, required + 1):
        col = f"node_{i}_reviewer_completion_time"
        filled += int(_is_time_filled(row.get(col, np.nan)))

    return required, filled, bool(filled == required)


def _to_success_bool(v: object) -> object:
    if isinstance(v, (bool, np.bool_)):
        return bool(v)
    if pd.isna(v):
        return np.nan
    s = str(v).strip().lower()
    if s in {"true", "1", "yes", "y"}:
        return True
    if s in {"false", "0", "no", "n"}:
        return False
    return np.nan


def _confusion_label(sim_success: object, pred_success: object) -> str:
    if pd.isna(sim_success) or pd.isna(pred_success):
        return "unknown"
    if bool(sim_success) and bool(pred_success):
        return "tp"
    if (not bool(sim_success)) and bool(pred_success):
        return "fp"
    if bool(sim_success) and (not bool(pred_success)):
        return "fn"
    return "tn"


for _df_name in ["VIZ_DF_ALL", "VIZ_DF"]:
    if _df_name not in globals() or globals()[_df_name].empty:
        continue

    _df = globals()[_df_name].copy()
    _df["sim_success_bool"] = _df["success"].map(_to_success_bool)

    for spec in MODEL_SPECS:
        model_key = spec["key"]
        triplets = _df.apply(
            lambda r, _prefix=spec["completion_prefix"]: _pred_success_from_completion_times(r, _prefix),
            axis=1,
        )
        _df[f"{model_key}_required_slots"] = triplets.map(lambda t: t[0])
        _df[f"{model_key}_filled_slots"] = triplets.map(lambda t: t[1])
        _df[f"{model_key}_pred_success"] = triplets.map(lambda t: t[2])

    reviewer_triplets = _df.apply(_pred_success_from_reviewer_times, axis=1)
    _df["reviewer_required_slots"] = reviewer_triplets.map(lambda t: t[0])
    _df["reviewer_filled_slots"] = reviewer_triplets.map(lambda t: t[1])

    # Treat reviewer-derived prediction as valid only for reviewed episodes.
    reviewed_mask = _df["node_1_reviewer_completion_time"].map(_is_time_filled)
    _df["reviewed_episode"] = reviewed_mask
    _df["reviewer_pred_success"] = reviewer_triplets.map(lambda t: t[2]).where(reviewed_mask, np.nan)

    for spec in MODEL_SPECS:
        model_key = spec["key"]
        _df[f"{model_key}_confusion"] = _df.apply(
            lambda r, _pred_col=f"{model_key}_pred_success": _confusion_label(
                r.get("sim_success_bool", np.nan),
                r.get(_pred_col, np.nan),
            ),
            axis=1,
        )

    _df["reviewer_confusion"] = _df.apply(
        lambda r: _confusion_label(r.get("sim_success_bool", np.nan), r.get("reviewer_pred_success", np.nan)),
        axis=1,
    )

    for model_key in [s["key"] for s in MODEL_SPECS] + ["reviewer"]:
        _df[f"{model_key}_tp"] = _df[f"{model_key}_confusion"].eq("tp")
        _df[f"{model_key}_fp"] = _df[f"{model_key}_confusion"].eq("fp")
        _df[f"{model_key}_tn"] = _df[f"{model_key}_confusion"].eq("tn")
        _df[f"{model_key}_fn"] = _df[f"{model_key}_confusion"].eq("fn")

    globals()[_df_name] = _df


MODEL_LABELS = {spec["key"]: spec["label"] for spec in MODEL_SPECS}


def _confusion_summary_table(df: pd.DataFrame, model_key: str) -> pd.DataFrame:
    conf_col = f"{model_key}_confusion"
    pred_col = f"{model_key}_pred_success"

    base = df[df["sim_success_bool"].isin([True, False])].copy()
    if model_key == "reviewer":
        base = base[base["reviewed_episode"].eq(True)].copy()

    counts = base[conf_col].value_counts().reindex(["tp", "fp", "tn", "fn"], fill_value=0)

    tp = int(counts["tp"])
    fp = int(counts["fp"])
    tn = int(counts["tn"])
    fn = int(counts["fn"])

    precision = tp / (tp + fp) if (tp + fp) else np.nan
    recall = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else np.nan

    return pd.DataFrame(
        {
            "model": [MODEL_LABELS.get(model_key, model_key)],
            "episodes_eval": [int(len(base))],
            "tp": [tp],
            "fp": [fp],
            "tn": [tn],
            "fn": [fn],
            "precision": [precision],
            "recall": [recall],
            "specificity": [specificity],
            "accuracy": [accuracy],
            "pred_success_rate": [float(base[pred_col].mean()) if len(base) else np.nan],
            "sim_success_rate": [float(base["sim_success_bool"].mean()) if len(base) else np.nan],
        }
    )


base_viz = VIZ_DF.copy()
base_viz = base_viz[base_viz["sim_success_bool"].isin([True, False])].copy()

summary_frames = [_confusion_summary_table(base_viz, spec["key"]) for spec in MODEL_SPECS]
summary_frames.append(_confusion_summary_table(base_viz, "reviewer"))
CONFUSION_SUMMARY = pd.concat(summary_frames, ignore_index=True)

agg_map: dict[str, tuple[str, str]] = {
    "episodes": ("episode_dir", "size"),
    "reviewer_pred_success_rate": ("reviewer_pred_success", "mean"),
}
for spec in MODEL_SPECS:
    agg_map[f"{spec['key']}_pred_success_rate"] = (f"{spec['key']}_pred_success", "mean")

SUCCESS_GROUP_TABLE = (
    base_viz.groupby("sim_success_bool", dropna=False)
    .agg(**agg_map)
    .reset_index()
    .rename(columns={"sim_success_bool": "sim_success"})
)

error_cases_by_model: dict[str, pd.DataFrame] = {}
for spec in MODEL_SPECS:
    mk = spec["key"]
    err = base_viz[base_viz[f"{mk}_confusion"].isin(["fp", "fn"])].copy()
    err[f"node_fill_ratio_{mk}"] = err[f"{mk}_filled_slots"] / err[f"{mk}_required_slots"].replace(0, np.nan)
    error_cases_by_model[mk] = err

ERROR_CASES_REVIEWER = base_viz[
    base_viz["reviewed_episode"].eq(True) & base_viz["reviewer_confusion"].isin(["fp", "fn"])
].copy()
ERROR_CASES_REVIEWER["node_fill_ratio_reviewer"] = (
    ERROR_CASES_REVIEWER["reviewer_filled_slots"]
    / ERROR_CASES_REVIEWER["reviewer_required_slots"].replace(0, np.nan)
)

display(Markdown("### Success Groups (sim success=True vs False)"))
display(SUCCESS_GROUP_TABLE)

display(Markdown("### Confusion Summary by Model"))
display(CONFUSION_SUMMARY)

for spec in MODEL_SPECS:
    mk = spec["key"]
    display(Markdown(f"### {spec['label']} Error Cases (FP/FN)"))
    display(
        error_cases_by_model[mk][
            [
                "scenario_id",
                "episode_dir",
                "episode_index",
                "bt_action_nodes",
                "sim_success_bool",
                f"{mk}_confusion",
                f"{mk}_filled_slots",
                f"{mk}_required_slots",
                f"node_fill_ratio_{mk}",
            ]
        ].head(30)
    )

display(Markdown("### Reviewer Error Cases (FP/FN, reviewed episodes only)"))
display(
    ERROR_CASES_REVIEWER[
        [
            "scenario_id",
            "episode_dir",
            "episode_index",
            "bt_action_nodes",
            "sim_success_bool",
            "reviewer_confusion",
            "reviewer_filled_slots",
            "reviewer_required_slots",
            "node_fill_ratio_reviewer",
        ]
    ].head(30)
)

# Visualization 1: grouped confusion counts.
plot_df = CONFUSION_SUMMARY.copy()
labels = ["tp", "fp", "tn", "fn"]
x = np.arange(len(plot_df))
width = 0.18

fig, ax = plt.subplots(figsize=(11.0, 4.8))
for idx, lab in enumerate(labels):
    ax.bar(x + (idx - 1.5) * width, plot_df[lab], width=width, label=lab.upper())

ax.set_xticks(x)
ax.set_xticklabels(plot_df["model"])
ax.set_xlabel("model")
ax.set_ylabel("episode count")
ax.set_title("Confusion Counts by Model (Episodes 2-21)")
ax.legend(loc="best")
plt.tight_layout()
plt.show()

# Visualization 2: prediction success rate by sim-success group.
rate_plot = SUCCESS_GROUP_TABLE.copy()
rate_plot["sim_success"] = rate_plot["sim_success"].map({True: "success=True", False: "success=False"})

rate_cols = [f"{spec['key']}_pred_success_rate" for spec in MODEL_SPECS] + ["reviewer_pred_success_rate"]
rate_labels = [spec["label"] for spec in MODEL_SPECS] + ["reviewer"]

x2 = np.arange(len(rate_plot))
n_bars = len(rate_cols)
w2 = 0.85 / max(n_bars, 1)
fig, ax = plt.subplots(figsize=(10.0, 4.8))
for idx, (col, lab) in enumerate(zip(rate_cols, rate_labels)):
    center_offset = idx - (n_bars - 1) / 2
    ax.bar(x2 + center_offset * w2, rate_plot[col], width=w2, label=lab)

ax.set_xticks(x2)
ax.set_xticklabels(rate_plot["sim_success"])
ax.set_ylim(0, 1)
ax.set_xlabel("sim success group")
ax.set_ylabel("predicted-success rate")
ax.set_title("Prediction Success Rate by Sim Outcome")
ax.legend(loc="best")
plt.tight_layout()
plt.show()

### Traditional Confusion Matrices and Reviewed Node-Similarity Checks

This section adds standard 2x2 confusion matrices:
- Rows = actual simulator outcome (`sim_success_bool`)
- Columns = predicted outcome (completion-time proxy)

Views included:
- Overall matrices for qwen3, qwen3.5, and qwen3.5 mod1 (all episodes in `VIZ_DF`).
- Reviewed-only matrix comparing actual success vs reviewer-derived success (`reviewer_pred_success`), where reviewer success requires all required `node_i_reviewer_completion_time` fields for `i=1..bt_action_nodes`.
- Reviewed-only reviewer-correct subset (`reviewer_confusion` in `tp`/`tn`) with node-similarity comparison between reviewer completed nodes and model completed nodes.
- Per-scenario matrices (up to 3 scenarios) for qwen3, qwen3.5, and qwen3.5 mod1 across all episodes (not review-filtered).

In [ ]:
def _plot_confusion_matrix_heatmap(
    ax: plt.Axes,
    df: pd.DataFrame,
    pred_col: str,
    title: str,
) -> pd.DataFrame:
    valid = df[df["sim_success_bool"].isin([True, False])].copy()
    matrix = (
        pd.crosstab(
            valid["sim_success_bool"].map({True: "Actual: Success", False: "Actual: Failure"}),
            valid[pred_col].map({True: "Pred: Success", False: "Pred: Failure"}),
            dropna=False,
        )
        .reindex(index=["Actual: Success", "Actual: Failure"], columns=["Pred: Success", "Pred: Failure"], fill_value=0)
        .astype(int)
    )

    im = ax.imshow(matrix.values, cmap="Blues", aspect="equal")
    ax.set_xticks(np.arange(matrix.shape[1]))
    ax.set_yticks(np.arange(matrix.shape[0]))
    ax.set_xticklabels(matrix.columns)
    ax.set_yticklabels(matrix.index)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

    max_v = matrix.values.max() if matrix.size else 0
    for r in range(matrix.shape[0]):
        for c in range(matrix.shape[1]):
            v = int(matrix.iloc[r, c])
            color = "white" if v > max_v / 2 else "black"
            ax.text(c, r, str(v), ha="center", va="center", color=color, fontsize=10, fontweight="bold")

    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    return matrix


model_specs_local = globals().get(
    "MODEL_SPECS",
    [
        {"key": "qwen3", "label": "qwen3"},
        {"key": "qwen35", "label": "qwen3.5"},
        {"key": "qwen35_mod1", "label": "qwen3.5 mod1"},
    ],
)

# 1) Overall confusion matrices on VIZ_DF (episodes 2-21)
overall_base = VIZ_DF[VIZ_DF["sim_success_bool"].isin([True, False])].copy()

n_models = len(model_specs_local)
fig, axes = plt.subplots(1, n_models, figsize=(6.0 * n_models, 4.8))
if n_models == 1:
    axes = [axes]

overall_mats: dict[str, pd.DataFrame] = {}
for idx, spec in enumerate(model_specs_local):
    mk = spec["key"]
    lab = spec["label"]
    pred_col = f"{mk}_pred_success"
    overall_mats[mk] = _plot_confusion_matrix_heatmap(
        axes[idx],
        overall_base,
        pred_col,
        f"{lab} Confusion Matrix (All Episodes 2-21)",
    )

plt.tight_layout()
plt.show()

display(Markdown("### Overall Confusion Matrices (counts)"))
for spec in model_specs_local:
    display(Markdown(f"#### {spec['label']}"))
    display(overall_mats[spec["key"]])


# 2) Reviewed-only subset based on node_1_reviewer_completion_time,
# compared using reviewer-derived success proxy.
reviewed_mask = VIZ_DF["node_1_reviewer_completion_time"].map(_is_time_filled)
reviewed_base = VIZ_DF[reviewed_mask & VIZ_DF["sim_success_bool"].isin([True, False])].copy()

display(Markdown(f"### Reviewed-Only Subset Size (node_1 reviewer time present): **{len(reviewed_base)}** rows"))

if reviewed_base.empty:
    display(Markdown("No reviewed rows found using `node_1_reviewer_completion_time` as the filter."))
else:
    fig, ax = plt.subplots(1, 1, figsize=(6.4, 5.0))
    cm_reviewed_reviewer = _plot_confusion_matrix_heatmap(
        ax,
        reviewed_base,
        "reviewer_pred_success",
        "Reviewer Confusion Matrix (Reviewed-Only)",
    )
    plt.tight_layout()
    plt.show()

    display(Markdown("#### Reviewed-only confusion matrix (actual vs reviewer success)"))
    display(cm_reviewed_reviewer)


# 3) Per-scenario confusion matrices across ALL episodes (not review-filtered),
# shown for up to 3 scenarios by coverage.
scenario_order = (
    overall_base.groupby("scenario_id", dropna=False)["episode_dir"]
    .nunique()
    .sort_values(ascending=False)
    .index.tolist()
)
scenario_subset = scenario_order[:3]

display(Markdown("### All-Episode Confusion Matrices by Scenario (up to 3 scenarios)"))
display(Markdown(f"Scenarios shown: **{', '.join(map(str, scenario_subset))}**"))

for sc in scenario_subset:
    sc_df = overall_base[overall_base["scenario_id"] == sc].copy()
    if sc_df.empty:
        continue

    fig, axes = plt.subplots(1, n_models, figsize=(6.0 * n_models, 4.6))
    if n_models == 1:
        axes = [axes]

    for idx, spec in enumerate(model_specs_local):
        mk = spec["key"]
        lab = spec["label"]
        _ = _plot_confusion_matrix_heatmap(
            axes[idx],
            sc_df,
            f"{mk}_pred_success",
            f"{lab} All-Episodes | {scenario_label(sc)}",
        )

    plt.tight_layout()
    plt.show()

### Completion Threshold Audit: All Nodes vs All-But-Last Node

This section compares simulator success against two completion-threshold criteria for each model:
- All nodes complete: filled completion-time slots equals required `bt_action_nodes`.
- All but last node complete: filled completion-time slots equals `bt_action_nodes - 1`.

Outputs include:
- Overall summary across episodes 2-21 for all three models.
- Scenario breakdown for the top 3 scenarios by coverage.
- Confusion-style counts (`tp`, `fp`, `tn`, `fn`) for each threshold criterion.

In [ ]:
threshold_specs = globals().get("MODEL_SPECS", [
    {"key": "qwen3", "label": "qwen3"},
    {"key": "qwen35", "label": "qwen3.5"},
    {"key": "qwen35_mod1", "label": "qwen3.5 mod1"},
])

def _threshold_metrics(df: pd.DataFrame, model_key: str, criterion: str, model_label: str) -> pd.DataFrame:
    req_col = f"{model_key}_required_slots"
    fill_col = f"{model_key}_filled_slots"
    if req_col not in df.columns or fill_col not in df.columns:
        return pd.DataFrame()

    base = df[df["sim_success_bool"].isin([True, False])].copy()
    if base.empty:
        return pd.DataFrame()

    req = pd.to_numeric(base[req_col], errors="coerce")
    filled = pd.to_numeric(base[fill_col], errors="coerce")

    if criterion == "all_nodes_complete":
        pred = req.gt(0) & filled.eq(req)
    else:
        pred = req.ge(2) & filled.eq(req - 1)

    actual = base["sim_success_bool"].astype(bool)
    tp = int((pred & actual).sum())
    fp = int((pred & ~actual).sum())
    tn = int((~pred & ~actual).sum())
    fn = int((~pred & actual).sum())

    met = base[pred]
    not_met = base[~pred]

    return pd.DataFrame(
        {
            "model": [model_label],
            "criterion": [criterion],
            "episodes_eval": [int(len(base))],
            "criterion_met_episodes": [int(pred.sum())],
            "criterion_not_met_episodes": [int((~pred).sum())],
            "success_rate_when_met": [float(met["sim_success_bool"].mean()) if len(met) else np.nan],
            "success_rate_when_not_met": [float(not_met["sim_success_bool"].mean()) if len(not_met) else np.nan],
            "tp": [tp],
            "fp": [fp],
            "tn": [tn],
            "fn": [fn],
        }
    )

threshold_base = VIZ_DF[VIZ_DF["sim_success_bool"].isin([True, False])].copy()
criteria = ["all_nodes_complete", "all_but_last_complete"]

overall_parts = []
for spec in threshold_specs:
    for crit in criteria:
        part = _threshold_metrics(threshold_base, spec["key"], crit, spec["label"])
        if not part.empty:
            overall_parts.append(part)
THRESHOLD_OVERALL = pd.concat(overall_parts, ignore_index=True) if overall_parts else pd.DataFrame()

scenario_order = (
    threshold_base.groupby("scenario_id", dropna=False)["episode_dir"]
    .nunique()
    .sort_values(ascending=False)
    .index.tolist()
)
scenario_top3 = scenario_order[:3]

scenario_parts = []
for sc in scenario_top3:
    sc_df = threshold_base[threshold_base["scenario_id"] == sc].copy()
    for spec in threshold_specs:
        for crit in criteria:
            part = _threshold_metrics(sc_df, spec["key"], crit, spec["label"])
            if not part.empty:
                part.insert(0, "scenario_id", sc)
                part.insert(1, "scenario_label", scenario_label(sc))
                scenario_parts.append(part)
THRESHOLD_BY_SCENARIO = pd.concat(scenario_parts, ignore_index=True) if scenario_parts else pd.DataFrame()

display(Markdown("### Overall Episodes: All-Nodes vs All-But-Last Thresholds"))
display(THRESHOLD_OVERALL)

display(Markdown(f"### Scenario Breakdown (Top 3): **{', '.join(map(str, scenario_top3))}**"))
display(THRESHOLD_BY_SCENARIO)

if not THRESHOLD_OVERALL.empty:
    p = THRESHOLD_OVERALL.pivot(index="model", columns="criterion", values="success_rate_when_met")
    p = p[[c for c in criteria if c in p.columns]]
    ax = p.plot(kind="bar", figsize=(8.8, 4.6), rot=0)
    ax.set_ylim(0, 1)
    ax.set_xlabel("model")
    ax.set_ylabel("sim success rate")
    ax.set_title("Success rate when threshold is met (overall)")
    ax.legend(title="criterion", loc="best")
    plt.tight_layout()
    plt.show()

if not THRESHOLD_BY_SCENARIO.empty:
    for crit in criteria:
        g = THRESHOLD_BY_SCENARIO[THRESHOLD_BY_SCENARIO["criterion"] == crit].copy()
        if g.empty:
            continue
        pivot_g = g.pivot(index="scenario_label", columns="model", values="success_rate_when_met")
        ax = pivot_g.plot(kind="bar", figsize=(10.0, 4.8), rot=20)
        ax.set_ylim(0, 1)
        ax.set_xlabel("scenario")
        ax.set_ylabel("sim success rate")
        ax.set_title(f"Success rate when {crit} is met (top 3 scenarios)")
        ax.legend(title="model", loc="best")
        plt.tight_layout()
        plt.show()


## Success Alignment Audit: TP/FP/FN/TN by Completion-Time Coverage

This section evaluates mission-engine predictions against simulator `success` labels in `VIZ_DF`.

Prediction rule per episode and per model:
- Let `n = bt_action_nodes`.
- Predicted success is `True` only if every required completion-time field for nodes `1..n` is present.

Columns added for reuse:
- `qwen3_pred_success`, `qwen35_pred_success`, `qwen35_mod1_pred_success`
- `qwen3_confusion`, `qwen35_confusion`, `qwen35_mod1_confusion`
- `qwen3_tp/fp/tn/fn`, `qwen35_tp/fp/tn/fn`, `qwen35_mod1_tp/fp/tn/fn`
- Slot-count diagnostics for each model (`*_required_slots`, `*_filled_slots`)

In [ ]:
# Reviewer node-status vs VLM comparison across models
# Requested columns:
# Model | Reviewed node events | Effective agreement | Strict agreement | Primary disagreement mode

if 'COMPARE_DF' not in globals() or COMPARE_DF.empty:
    display(Markdown('### COMPARE_DF is unavailable or empty. Run earlier ingestion cells first.'))
else:
    all_models = pd.DataFrame({'model_name': sorted(COMPARE_DF['model_name'].dropna().astype(str).unique())})

    reviewed = COMPARE_DF.copy()

    # Keep rows that appear to have reviewer-side annotation evidence.
    reviewer_status_clean = reviewed['reviewer_status'].fillna('').astype(str).str.strip()
    reviewer_agree_clean = reviewed['reviewer_agree'].fillna('').astype(str).str.strip().str.lower()
    reviewer_time_present = reviewed['reviewer_video_time_s'].notna() if 'reviewer_video_time_s' in reviewed.columns else False

    reviewed_mask = reviewer_status_clean.ne('') | reviewer_agree_clean.ne('') | reviewer_time_present
    reviewed = reviewed[reviewed_mask].copy()

    if reviewed.empty:
        out = all_models.copy()
        out['reviewed_node_events'] = 0
        out['effective_agreement_rate_pct'] = np.nan
        out['strict_agreement_rate_pct'] = np.nan
        out['primary_disagreement_mode'] = 'none'
    else:
        # Effective agreement follows notebook convention: y / p / timing_aligned.
        effective_ok = reviewed['agreement_effective'].fillna('').astype(str).str.lower().isin(['y', 'p', 'timing_aligned'])
        strict_ok = reviewer_agree_clean.eq('y')

        summary = (
            reviewed.assign(_effective_ok=effective_ok.astype(int), _strict_ok=strict_ok.astype(int))
            .groupby('model_name', dropna=False)
            .agg(
                reviewed_node_events=('model_name', 'size'),
                effective_agreement_rate_pct=('_effective_ok', 'mean'),
                strict_agreement_rate_pct=('_strict_ok', 'mean'),
            )
            .reset_index()
        )

        # Primary disagreement mode = most frequent model_status_norm among disagreement rows.
        dis = reviewed[reviewed['agreement_effective'].fillna('').astype(str).str.lower().eq('n')].copy()
        if dis.empty:
            primary = pd.DataFrame({'model_name': summary['model_name'], 'primary_disagreement_mode': 'none'})
        else:
            mode_counts = (
                dis.groupby(['model_name', 'model_status_norm'], dropna=False)
                .size()
                .reset_index(name='count')
            )
            top = mode_counts.sort_values(['model_name', 'count'], ascending=[True, False]).groupby('model_name', as_index=False).head(1)
            top['primary_disagreement_mode'] = top['model_status_norm'].fillna('unknown').astype(str)
            primary = top[['model_name', 'primary_disagreement_mode']]

        out = all_models.merge(summary, on='model_name', how='left').merge(primary, on='model_name', how='left')
        out['reviewed_node_events'] = out['reviewed_node_events'].fillna(0).astype(int)
        out['primary_disagreement_mode'] = out['primary_disagreement_mode'].fillna('none')

    model_display_map = {
        'qwen3': 'Qwen3',
        'qwen3.5': 'Qwen3.5',
        'qwen3.5_tailored': 'Qwen3.5 tailored',
        'qwen3.5_mod1': 'Qwen3.5 mod1',
    }
    out['Model'] = out['model_name'].map(lambda x: model_display_map.get(str(x), str(x)))
    out['Reviewed node events'] = out['reviewed_node_events'].astype(int)
    out['Effective agreement'] = (100.0 * out['effective_agreement_rate_pct']).round(1).map(
        lambda v: 'n/a' if pd.isna(v) else f'{v:.1f}%'
    )
    out['Strict agreement'] = (100.0 * out['strict_agreement_rate_pct']).round(1).map(
        lambda v: 'n/a' if pd.isna(v) else f'{v:.1f}%'
    )
    out['Primary disagreement mode'] = out['primary_disagreement_mode']

    # Keep requested output schema.
    REVIEWER_VLM_MODEL_COMPARE = out[
        [
            'Model',
            'Reviewed node events',
            'Effective agreement',
            'Strict agreement',
            'Primary disagreement mode',
        ]
    ].sort_values('Model').reset_index(drop=True)

    display(Markdown('### Reviewer Node Status vs VLM by Model'))
    display(REVIEWER_VLM_MODEL_COMPARE)

In [ ]:
MASTER_EXPORT_DF = build_episode_master_export_table(
    logs_dir=LOGS_DIR,
    schedule_path=SCHEDULE_PATH,
    only_schedule_complete=True,
    max_node_slots=4,
    export_csv=True,
    export_path=LOGS_DIR / 'episode_master_export.csv',
)

if MASTER_EXPORT_DF.empty:
    display(Markdown('### Master export table is empty.'))
else:
    MASTER_EXPORT_DF = MASTER_EXPORT_DF.copy()
    MASTER_EXPORT_DF['episode_idx'] = pd.to_numeric(MASTER_EXPORT_DF['episode_idx'], errors='coerce')
    MASTER_EXPORT_2_21 = MASTER_EXPORT_DF[
        MASTER_EXPORT_DF['episode_idx'].between(2, 21, inclusive='both')
    ].copy()

    display(Markdown('## Master Episode Export (Preview, Episodes 2-21)'))
    display(
        MASTER_EXPORT_2_21[
            [
                'scenario_id',
                'episode_dir',
                'episode_idx',
                'total_bt_action_nodes',
                'reviewer_bt_nodes_complete',
                'qwen3_bt_nodes_complete',
                'qwen35_bt_nodes_complete',
                'reviewer_node_overflow_count',
                'qwen3_node_overflow_count',
                'qwen35_node_overflow_count',
            ]
        ].head(12)
    )
    display(Markdown(f"Master export shape (all schedule-complete episodes): **{MASTER_EXPORT_DF.shape[0]} rows x {MASTER_EXPORT_DF.shape[1]} columns**"))
    display(Markdown(f"CSV export path: `{MASTER_EXPORT_DF.attrs.get('export_path', str(LOGS_DIR / 'episode_master_export.csv'))}`"))

In [ ]:
display(Markdown('## Master Export Column Map'))

if 'MASTER_EXPORT_DF' not in globals() or MASTER_EXPORT_DF.empty:
    display(Markdown('Master export dataframe is not available yet. Run the previous cell first.'))
else:
    cols = MASTER_EXPORT_DF.columns.tolist()

    section_cols = {
        'episode_identity': [
            c for c in [
                'episode_dir',
                'episode_id_meta',
                'scenario_id',
                'scenario_label',
                'run_id',
                'episode_idx',
                'sim_success',
                'sim_success_bool',
                'episode_has_review',
                'schedule_complete',
            ]
            if c in cols
        ],
        'episode_metrics': [
            c for c in [
                'total_bt_action_nodes',
                'reviewer_bt_nodes_complete',
                'qwen3_bt_nodes_complete',
                'qwen35_bt_nodes_complete',
                'reviewer_all_nodes_time_filled',
                'qwen3_all_nodes_complete',
                'qwen35_all_nodes_complete',
                'reviewer_node_completion_rate_pct',
                'qwen3_node_completion_rate_pct',
                'qwen35_node_completion_rate_pct',
                'reviewer_completion_times_s_all',
                'qwen3_completion_times_s_all',
                'qwen35_completion_times_s_all',
                'reviewer_completion_times_mss_all',
                'canonical_action_node_ids_json',
            ]
            if c in cols
        ],
        'reviewer_slots': [c for c in cols if c.startswith('reviewer_node_') and '_overflow_' not in c],
        'qwen3_slots': [c for c in cols if c.startswith('qwen3_node_') and '_overflow_' not in c],
        'qwen35_slots': [c for c in cols if c.startswith('qwen35_node_') and '_overflow_' not in c],
        'overflow': [c for c in cols if '_overflow_count' in c or '_overflow_json' in c],
        'compare_metrics': [
            c for c in cols
            if c.startswith('qwen3_compare_')
            or c.startswith('qwen35_compare_')
            or c.startswith('qwen3_agreement_')
            or c.startswith('qwen35_agreement_')
            or c.startswith('qwen3_timing_')
            or c.startswith('qwen35_timing_')
        ],
    }

    summary_rows = [
        {'section': k, 'column_count': len(v)}
        for k, v in section_cols.items()
    ]
    display(pd.DataFrame(summary_rows))

    section_table = pd.DataFrame(
        [
            {'section': k, 'columns': ', '.join(v)}
            for k, v in section_cols.items()
        ]
    )
    display(section_table)

In [ ]:
def _clean_text(v: object) -> str:
    return '' if pd.isna(v) else str(v).strip().lower()


def binom_ci_95(successes: int, n: int) -> tuple[float, float]:
    if n <= 0:
        return (np.nan, np.nan)
    p = successes / n
    se = np.sqrt(p * (1.0 - p) / n)
    lo = max(0.0, p - 1.96 * se)
    hi = min(1.0, p + 1.96 * se)
    return (lo, hi)


def disagreement_bucket(row: pd.Series) -> str:
    status_txt = _clean_text(row.get('reviewer_status_clean', ''))
    notes_txt = _clean_text(row.get('reviewer_notes', ''))
    combined = f'{status_txt} {notes_txt}'.strip()

    if status_txt == '':
        if pd.notna(row.get('timing_abs_delta_s')) and float(row.get('timing_abs_delta_s')) > 2.0:
            return 'time_only_mismatch_gt2s'
        return 'time_only_mismatch_unspecified'

    if any(k in combined for k in ['next node', 'wrong node', 'different node', 'node mismatch', 'advance', 'should be next', 'stuck on node', 'previous node']):
        return 'node_transition_mismatch'
    if any(k in combined for k in ['fail', 'failed', 'error', 'abort', 'timeout', 'stuck']):
        return 'failure_signal_mismatch'
    if any(k in combined for k in ['complete', 'completed', 'done', 'success']):
        return 'completion_criterion_mismatch'
    if any(k in combined for k in ['running', 'progress', 'ongoing', 'in progress']):
        return 'progress_state_mismatch'
    return 'other_status_override'


BENCHMARK_DF = pd.DataFrame([
    {'task': 'gr1_unified/PnPBottleToCabinetClose_GR1ArmsAndWaistFourierHands_Env', 'benchmark_success_rate_pct': 51.5},
    {'task': 'gr1_unified/PnPCanToDrawerClose_GR1ArmsAndWaistFourierHands_Env', 'benchmark_success_rate_pct': 13.0},
    {'task': 'gr1_unified/PnPCupToDrawerClose_GR1ArmsAndWaistFourierHands_Env', 'benchmark_success_rate_pct': 8.5},
    {'task': 'gr1_unified/PnPMilkToMicrowaveClose_GR1ArmsAndWaistFourierHands_Env', 'benchmark_success_rate_pct': 14.0},
    {'task': 'gr1_unified/PnPPotatoToMicrowaveClose_GR1ArmsAndWaistFourierHands_Env', 'benchmark_success_rate_pct': 41.5},
    {'task': 'gr1_unified/PnPWineToCabinetClose_GR1ArmsAndWaistFourierHands_Env', 'benchmark_success_rate_pct': 16.5},
    {'task': 'gr1_unified/PosttrainPnPNovelFromCuttingboardToBasketSplitA_GR1ArmsAndWaistFourierHands_Env', 'benchmark_success_rate_pct': 58.0},
    {'task': 'gr1_unified/PosttrainPnPNovelFromCuttingboardToCardboardboxSplitA_GR1ArmsAndWaistFourierHands_Env', 'benchmark_success_rate_pct': 46.5},
    {'task': 'gr1_unified/PosttrainPnPNovelFromCuttingboardToPanSplitA_GR1ArmsAndWaistFourierHands_Env', 'benchmark_success_rate_pct': 68.5},
    {'task': 'gr1_unified/PosttrainPnPNovelFromCuttingboardToPotSplitA_GR1ArmsAndWaistFourierHands_Env', 'benchmark_success_rate_pct': 65.0},
    {'task': 'gr1_unified/PosttrainPnPNovelFromCuttingboardToTieredbasketSplitA_GR1ArmsAndWaistFourierHands_Env', 'benchmark_success_rate_pct': 46.5},
    {'task': 'gr1_unified/PosttrainPnPNovelFromPlacematToBasketSplitA_GR1ArmsAndWaistFourierHands_Env', 'benchmark_success_rate_pct': 58.5},
    {'task': 'gr1_unified/PosttrainPnPNovelFromPlacematToBowlSplitA_GR1ArmsAndWaistFourierHands_Env', 'benchmark_success_rate_pct': 57.5},
    {'task': 'gr1_unified/PosttrainPnPNovelFromPlacematToPlateSplitA_GR1ArmsAndWaistFourierHands_Env', 'benchmark_success_rate_pct': 63.0},
    {'task': 'gr1_unified/PosttrainPnPNovelFromPlacematToTieredshelfSplitA_GR1ArmsAndWaistFourierHands_Env', 'benchmark_success_rate_pct': 28.5},
    {'task': 'gr1_unified/PosttrainPnPNovelFromPlateToBowlSplitA_GR1ArmsAndWaistFourierHands_Env', 'benchmark_success_rate_pct': 57.0},
    {'task': 'gr1_unified/PosttrainPnPNovelFromPlateToCardboardboxSplitA_GR1ArmsAndWaistFourierHands_Env', 'benchmark_success_rate_pct': 43.5},
    {'task': 'gr1_unified/PosttrainPnPNovelFromPlateToPanSplitA_GR1ArmsAndWaistFourierHands_Env', 'benchmark_success_rate_pct': 51.0},
    {'task': 'gr1_unified/PosttrainPnPNovelFromPlateToPlateSplitA_GR1ArmsAndWaistFourierHands_Env', 'benchmark_success_rate_pct': 78.7},
    {'task': 'gr1_unified/PosttrainPnPNovelFromTrayToCardboardboxSplitA_GR1ArmsAndWaistFourierHands_Env', 'benchmark_success_rate_pct': 51.5},
    {'task': 'gr1_unified/PosttrainPnPNovelFromTrayToPlateSplitA_GR1ArmsAndWaistFourierHands_Env', 'benchmark_success_rate_pct': 71.0},
    {'task': 'gr1_unified/PosttrainPnPNovelFromTrayToPotSplitA_GR1ArmsAndWaistFourierHands_Env', 'benchmark_success_rate_pct': 64.5},
    {'task': 'gr1_unified/PosttrainPnPNovelFromTrayToTieredbasketSplitA_GR1ArmsAndWaistFourierHands_Env', 'benchmark_success_rate_pct': 57.0},
    {'task': 'gr1_unified/PosttrainPnPNovelFromTrayToTieredshelfSplitA_GR1ArmsAndWaistFourierHands_Env', 'benchmark_success_rate_pct': 31.5},
])


def task_key(s: object) -> str:
    txt = '' if pd.isna(s) else str(s)
    txt = txt.split('/')[-1]
    txt = re.sub(r'_GR1.*$', '', txt)
    return txt.lower()


BENCHMARK_DF['task_key'] = BENCHMARK_DF['task'].map(task_key)
SCENARIO_MAP_DF = pd.DataFrame({'scenario_id': sorted(COMPARE_DF['scenario_id'].dropna().astype(str).unique())})
SCENARIO_MAP_DF['scenario_key'] = SCENARIO_MAP_DF['scenario_id'].map(task_key)
SCENARIO_MAP_DF['scenario_label'] = SCENARIO_MAP_DF['scenario_id'].map(scenario_label)
SCENARIO_MAP_DF = SCENARIO_MAP_DF.merge(BENCHMARK_DF[['task', 'task_key', 'benchmark_success_rate_pct']], left_on='scenario_key', right_on='task_key', how='left')

display(Markdown('### Benchmark Mapping Preview'))
display(SCENARIO_MAP_DF[['scenario_id', 'scenario_label', 'task', 'benchmark_success_rate_pct']])

## Section 1: Overview and Cross-Model Trends

This section summarizes:
- Data coverage and agreement distribution for the surveyed subset.
- Distribution across scenarios and runs (episodes 2-21 only).
- How simulator success compares with BT node-completion proxy for each mission engine.
- Where Qwen3 and Qwen3.5 differ in practical audit outcomes.

In [ ]:
OVERVIEW_TABLES = summarize_model_compare(COMPARE_DF)
COVERAGE_TABLE = OVERVIEW_TABLES['coverage'].copy()
AGREEMENT_TABLE = OVERVIEW_TABLES['agreement'].copy()
SIM_BT_TABLE = OVERVIEW_TABLES['sim_bt'].copy()

display(Markdown('### Coverage and Agreement Tables'))
display(COVERAGE_TABLE)
display(AGREEMENT_TABLE.sort_values(['model_name', 'agreement_effective']).reset_index(drop=True))
display(SIM_BT_TABLE.sort_values(['scenario_id', 'model_name']).reset_index(drop=True))

In [ ]:
display(Markdown('### Three-Model Comparison (qwen3 vs qwen3.5 vs qwen3.5 tailored)'))

model_order = ['qwen3', 'qwen3.5', 'qwen3.5_tailored']
coverage_three = COVERAGE_TABLE[COVERAGE_TABLE['model_name'].isin(model_order)].copy()
agreement_three = AGREEMENT_TABLE[AGREEMENT_TABLE['model_name'].isin(model_order)].copy()
sim_bt_three = SIM_BT_TABLE[SIM_BT_TABLE['model_name'].isin(model_order)].copy()

if coverage_three.empty:
    display(Markdown('No three-model coverage rows available yet.'))
else:
    coverage_plot = coverage_three.set_index('model_name').reindex(model_order).dropna(how='all')
    ax = coverage_plot[['episodes', 'rows']].plot(kind='bar', figsize=(8.8, 4.6), rot=20)
    ax.set_title('Coverage by model (episodes and node rows)')
    ax.set_xlabel('model')
    ax.set_ylabel('count')
    plt.tight_layout()
    plt.show()

if not agreement_three.empty:
    a = agreement_three.copy()
    a = a[a['agreement_effective'].isin(['y', 'p', 'timing_aligned', 'n'])]
    p = a.pivot(index='model_name', columns='agreement_effective', values='rate_pct').fillna(0.0)
    p = p.reindex(model_order).dropna(how='all')
    order = [c for c in ['y', 'p', 'timing_aligned', 'n'] if c in p.columns]
    if len(order) > 0:
        ax = p[order].plot(kind='bar', stacked=True, figsize=(9.2, 4.8), rot=20)
        ax.set_title('Agreement profile by model (effective labels)')
        ax.set_xlabel('model')
        ax.set_ylabel('rate (%)')
        plt.tight_layout()
        plt.show()

display(Markdown('### qwen3.5 baseline vs qwen3.5 tailored (did prompt tuning change outcomes?)'))

q35_pair = sim_bt_three[sim_bt_three['model_name'].isin(['qwen3.5', 'qwen3.5_tailored'])].copy()
if q35_pair.empty:
    display(Markdown('No qwen3.5 pair rows found in sim/BT summary.'))
else:
    pair_pivot = q35_pair.pivot(
        index='scenario_id',
        columns='model_name',
        values=['sim_success_rate_pct', 'bt_complete_rate_pct', 'sim_minus_bt_gap_pct'],
    )
    pair_pivot.columns = ['__'.join(col) for col in pair_pivot.columns]
    pair_pivot = pair_pivot.reset_index()
    pair_pivot['scenario_label'] = pair_pivot['scenario_id'].map(scenario_label)

    for metric in ['sim_success_rate_pct', 'bt_complete_rate_pct', 'sim_minus_bt_gap_pct']:
        base_col = f'{metric}__qwen3.5'
        tailored_col = f'{metric}__qwen3.5_tailored'
        delta_col = f'{metric}__delta_tailored_minus_base'
        if base_col in pair_pivot.columns and tailored_col in pair_pivot.columns:
            pair_pivot[delta_col] = (pair_pivot[tailored_col] - pair_pivot[base_col]).round(1)

    metric_cols = [
        c
        for c in [
            'sim_success_rate_pct__qwen3.5',
            'sim_success_rate_pct__qwen3.5_tailored',
            'sim_success_rate_pct__delta_tailored_minus_base',
            'bt_complete_rate_pct__qwen3.5',
            'bt_complete_rate_pct__qwen3.5_tailored',
            'bt_complete_rate_pct__delta_tailored_minus_base',
            'sim_minus_bt_gap_pct__qwen3.5',
            'sim_minus_bt_gap_pct__qwen3.5_tailored',
            'sim_minus_bt_gap_pct__delta_tailored_minus_base',
        ]
        if c in pair_pivot.columns
    ]
    display(pair_pivot[['scenario_id', 'scenario_label'] + metric_cols].sort_values('scenario_id').reset_index(drop=True))

    delta_cols = [
        c
        for c in [
            'sim_success_rate_pct__delta_tailored_minus_base',
            'bt_complete_rate_pct__delta_tailored_minus_base',
            'sim_minus_bt_gap_pct__delta_tailored_minus_base',
        ]
        if c in pair_pivot.columns
    ]
    if len(delta_cols) > 0:
        x = np.arange(len(pair_pivot))
        w = 0.24
        fig, ax = plt.subplots(figsize=(11.0, 4.8))
        for i, col in enumerate(delta_cols):
            ax.bar(x + (i - (len(delta_cols) - 1) / 2.0) * w, pair_pivot[col], width=w, label=col.replace('__delta_tailored_minus_base', ''))
        ax.axhline(0.0, color='gray', linestyle='--', linewidth=1.0)
        ax.set_xticks(x)
        ax.set_xticklabels(pair_pivot['scenario_label'], rotation=20, ha='right')
        ax.set_title('Tailored - baseline deltas for qwen3.5')
        ax.set_xlabel('scenario')
        ax.set_ylabel('delta (pct-points)')
        ax.legend(loc='best')
        plt.tight_layout()
        plt.show()

    # Episode-level agreement delta for the qwen3.5 pair.
    q35_episode = COMPARE_DF[COMPARE_DF['model_name'].isin(['qwen3.5', 'qwen3.5_tailored'])].copy()
    if not q35_episode.empty:
        q35_agree = (
            q35_episode[q35_episode['agreement_effective'].isin(['y', 'p', 'timing_aligned', 'n'])]
            .groupby(['model_name', 'scenario_id', 'agreement_effective'], dropna=False)
            .size()
            .reset_index(name='count')
        )
        totals = q35_agree.groupby(['model_name', 'scenario_id'])['count'].transform('sum')
        q35_agree['rate_pct'] = (100.0 * q35_agree['count'] / totals.replace(0, np.nan)).round(1)
        display(Markdown('Agreement distribution for qwen3.5 baseline vs tailored (by scenario)'))
        display(q35_agree.sort_values(['scenario_id', 'model_name', 'agreement_effective']).reset_index(drop=True))

In [ ]:
# Utility-provided baseline plots: agreement profile + timing distribution
plot_model_compare(AGREEMENT_TABLE, COMPARE_DF)

In [ ]:
display(Markdown('### Scenario Distribution Across Runs (Episodes 2-21)'))
dist_df = (
    COMPARE_DF[['scenario_id', 'run_id', 'episode_idx']]
    .drop_duplicates()
    .groupby('scenario_id', dropna=False)
    .agg(runs=('run_id', 'nunique'), episodes=('episode_idx', 'size'))
    .reset_index()
)
dist_df['scenario_label'] = dist_df['scenario_id'].map(scenario_label)
display(dist_df.sort_values('scenario_id').reset_index(drop=True))

plot_dist_df = dist_df.sort_values('scenario_id').reset_index(drop=True)
ax = plot_dist_df.set_index('scenario_label')['episodes'].plot(kind='bar', figsize=(8.5, 4.8), rot=20)
ax.set_title('Episode count by scenario (for a single run per scenario, episodes 2-21 only)')
ax.set_xlabel('scenario')
ax.set_ylabel('episodes (2-21 only)')
plt.tight_layout()
plt.show()

In [ ]:
display(Markdown('### Simulator Success vs Episode-Level BT/Reviewer Node Completion Rates'))

EPISODE_NODE_DF = build_episode_node_completion_dataframe(
    logs_dir=LOGS_DIR,
    schedule_path=SCHEDULE_PATH,
    only_schedule_complete=True,
)

EPISODE_NODE_TABLES = build_episode_completion_tables(EPISODE_NODE_DF)
EPISODE_NODE_DETAIL_DF = EPISODE_NODE_TABLES['episode_node_detail'].copy()
EPISODE_NODE_BACKBONE_DF = EPISODE_NODE_TABLES['episode_node_backbone'].copy()
EPISODE_MODEL_SUMMARY_DF = EPISODE_NODE_TABLES['episode_model_summary'].copy()
reviewed_table = EPISODE_NODE_TABLES['reviewed_scenario_rates'].copy()
all_episodes_table = EPISODE_NODE_TABLES['all_scenario_rates'].copy()

reviewed_episode_slots = int(EPISODE_MODEL_SUMMARY_DF.loc[EPISODE_MODEL_SUMMARY_DF['episode_has_review'], 'episode_dir'].nunique()) if not EPISODE_MODEL_SUMMARY_DF.empty else 0
all_episode_slots = int(EPISODE_MODEL_SUMMARY_DF['episode_dir'].nunique()) if not EPISODE_MODEL_SUMMARY_DF.empty else 0

display(Markdown('### Episode-Node Detail (long format: one row per node per model per episode)'))
display(EPISODE_NODE_DETAIL_DF)

display(Markdown('### Episode-Node Backbone (wide format: reviewer + qwen3 + qwen3.5 on each node row)'))
display(EPISODE_NODE_BACKBONE_DF)

display(Markdown('### Episode-Level Summary (reviewed + not reviewed, with node completion stats)'))
display(EPISODE_MODEL_SUMMARY_DF)

display(Markdown(
    '### Reviewed Episodes: Scenario-Level Detail (includes reviewer all-node completion-time metric)\n'
    f'- Unique reviewed episode slots in this filtered dataset: **{reviewed_episode_slots}**'
))
display(reviewed_table.sort_values(['scenario_id', 'model_name']).reset_index(drop=True))

for m in sorted(reviewed_table['model_name'].dropna().unique()):
    g = reviewed_table[reviewed_table['model_name'] == m].sort_values('scenario_id').reset_index(drop=True)
    if g.empty:
        continue
    x = np.arange(len(g))
    w = 0.26 
    fig, ax = plt.subplots(figsize=(10.2, 4.8))
    ax.bar(x - w, g['sim_success_rate_pct'], width=w, label='sim success rate')
    ax.bar(x, g['bt_all_nodes_complete_rate_pct'], width=w, label='all BT action nodes complete rate')
    ax.bar(x + w, g['reviewer_all_nodes_time_filled_rate_pct'], width=w, label='all reviewer BT node times filled rate')
    ax.set_xticks(x)
    ax.set_xticklabels(g['scenario_label'], rotation=20)
    ax.set_ylabel('rate (%)')
    ax.set_xlabel('scenario')
    ax.set_title(f'{m}: reviewed episodes only')
    ax.legend(loc='best')
    plt.tight_layout()
    plt.show()

display(Markdown(
    '### All Episode Slots: Scenario-Level Detail (no reviewer completion-time metric)\n'
    f'- Unique episode slots in this filtered dataset (target near 300): **{all_episode_slots}**'
))
display(all_episodes_table.sort_values(['scenario_id', 'model_name']).reset_index(drop=True))

### Section 1 Deep-Dive: Interpreting Sim Success vs BT Completion
Top-line simulator success and BT completion rates can hide different failure modes. The next cell decomposes episodes into four mutually exclusive buckets:
- `sim_and_bt`: simulator success and at least one completed BT node row both observed.
- `sim_only`: simulator success without BT-complete evidence.
- `bt_only`: BT-complete evidence despite simulator failure.
- `neither`: neither signal indicates success.

This makes the comparison more academically interpretable by attributing gaps to specific mechanism-level mismatch patterns rather than aggregate rates alone.

In [ ]:
EPISODE_MODEL_DF = (
    COMPARE_DF.groupby(['model_name', 'scenario_id', 'episode_dir'], dropna=False)
    .agg(
        sim_success=('sim_success', 'first'),
        bt_complete=('model_status_norm', lambda s: bool((pd.Series(s) == 'complete').any())),
        any_disagreement=('agreement_effective', lambda s: bool((pd.Series(s) == 'n').any())),
    )
    .reset_index()
)

EPISODE_MODEL_DF['sim_success_bool'] = EPISODE_MODEL_DF['sim_success'].eq(True)
EPISODE_MODEL_DF['outcome_bucket'] = np.select(
    [
        EPISODE_MODEL_DF['sim_success_bool'] & EPISODE_MODEL_DF['bt_complete'],
        EPISODE_MODEL_DF['sim_success_bool'] & (~EPISODE_MODEL_DF['bt_complete']),
        (~EPISODE_MODEL_DF['sim_success_bool']) & EPISODE_MODEL_DF['bt_complete'],
    ],
    ['sim_and_bt', 'sim_only', 'bt_only'],
    default='neither',
)

decomp = (
    EPISODE_MODEL_DF.groupby(['model_name', 'scenario_id', 'outcome_bucket'], dropna=False)
    .size()
    .reset_index(name='episodes')
)
tot = decomp.groupby(['model_name', 'scenario_id'])['episodes'].transform('sum')
decomp['rate_pct'] = (100.0 * decomp['episodes'] / tot.replace(0, np.nan)).round(1)
decomp['scenario_label'] = decomp['scenario_id'].map(scenario_label)

display(Markdown('### Deeper decomposition of simulator success vs BT complete proxy'))
display(decomp.sort_values(['model_name', 'scenario_id', 'outcome_bucket']).reset_index(drop=True))

for m in sorted(decomp['model_name'].dropna().unique()):
    g = decomp[decomp['model_name'] == m].copy()
    p = g.pivot(index='scenario_label', columns='outcome_bucket', values='rate_pct').fillna(0.0)
    order = [c for c in ['sim_and_bt', 'sim_only', 'bt_only', 'neither'] if c in p.columns]
    p = p[order]
    ax = p.plot(kind='bar', stacked=True, figsize=(10.2, 4.8), rot=20)
    ax.set_title(f'{m}: episode-level outcome decomposition')
    ax.set_xlabel('scenario')
    ax.set_ylabel('rate (%)')
    ax.legend(title='bucket', loc='best')
    plt.tight_layout()
    plt.show()

gap_df = (
    EPISODE_MODEL_DF.groupby(['model_name', 'scenario_id'], dropna=False)
    .agg(
        episodes=('episode_dir', 'nunique'),
        sim_success_rate_pct=('sim_success_bool', lambda s: round(100.0 * pd.Series(s).mean(), 1)),
        bt_complete_rate_pct=('bt_complete', lambda s: round(100.0 * pd.Series(s).mean(), 1)),
        disagreement_episode_rate_pct=('any_disagreement', lambda s: round(100.0 * pd.Series(s).mean(), 1)),
    )
    .reset_index()
)
gap_df['scenario_label'] = gap_df['scenario_id'].map(scenario_label)
gap_df['sim_minus_bt_gap_pct'] = (gap_df['sim_success_rate_pct'] - gap_df['bt_complete_rate_pct']).round(1)
display(gap_df.sort_values(['scenario_id', 'model_name']).reset_index(drop=True))

if not gap_df.empty:
    top_gap = gap_df.loc[gap_df['sim_minus_bt_gap_pct'].abs().idxmax()]
    direction = 'simulator > BT proxy' if top_gap['sim_minus_bt_gap_pct'] > 0 else 'BT proxy > simulator'
    display(Markdown(
        '### Section 1 Response: Why top-line rates can be misleading\n'
        f"- Largest absolute sim-vs-BT gap: **{top_gap['model_name']}** on **{top_gap['scenario_label']}** with **{top_gap['sim_minus_bt_gap_pct']} pct-points** ({direction}).\n"
        '- The decomposition table separates the gap into `sim_only` versus `bt_only` episodes, which distinguishes execution-level miss from BT-level over/under-claim.\n'
        '- The disagreement-episode rate adds a third axis: even when sim and BT both look positive, human disagreement can still be high.'
    ))

### Section 1 External Baseline Check: GR00T RoboCasa Posted Rates
To contextualize our observed simulator success, the next cell compares each matched scenario against posted GR1 benchmark rates and adds a simple 95% confidence interval around our observed rate. This allows a direct statement of whether differences are likely within sampling variance or outside an expected margin-of-error envelope.

In [ ]:
SIM_EPISODE_DF = (
    COMPARE_DF[['episode_dir', 'scenario_id', 'sim_success']]
    .drop_duplicates(subset=['episode_dir'])
    .copy()
)
SIM_EPISODE_DF['sim_success_bool'] = SIM_EPISODE_DF['sim_success'].eq(True)

our_rates = (
    SIM_EPISODE_DF.groupby('scenario_id', dropna=False)
    .agg(
        n_episodes=('episode_dir', 'nunique'),
        successes=('sim_success_bool', lambda s: int(pd.Series(s).sum())),
    )
    .reset_index()
)
our_rates['our_success_rate_pct'] = (100.0 * our_rates['successes'] / our_rates['n_episodes'].replace(0, np.nan)).round(1)
our_rates[['ci95_lo', 'ci95_hi']] = our_rates.apply(
    lambda r: pd.Series(binom_ci_95(int(r['successes']), int(r['n_episodes']))),
    axis=1,
)
our_rates['ci95_lo_pct'] = (100.0 * our_rates['ci95_lo']).round(1)
our_rates['ci95_hi_pct'] = (100.0 * our_rates['ci95_hi']).round(1)

bench_compare = our_rates.merge(
    SCENARIO_MAP_DF[['scenario_id', 'task', 'benchmark_success_rate_pct', 'scenario_label']],
    on='scenario_id',
    how='left',
)
bench_compare['sim_plus_benchmark_pct'] = (
    bench_compare['our_success_rate_pct'] + bench_compare['benchmark_success_rate_pct']
).round(1)
bench_compare['difference_sim_minus_benchmark_pct'] = (
    bench_compare['our_success_rate_pct'] - bench_compare['benchmark_success_rate_pct']
).round(1)
bench_compare['delta_vs_benchmark_pct'] = bench_compare['difference_sim_minus_benchmark_pct']
bench_compare['benchmark_within_our_ci95'] = (
    bench_compare['benchmark_success_rate_pct'].ge(bench_compare['ci95_lo_pct'])
    & bench_compare['benchmark_success_rate_pct'].le(bench_compare['ci95_hi_pct'])
)
bench_compare['outside_95moe_proxy'] = (
    bench_compare['difference_sim_minus_benchmark_pct'].abs() > (bench_compare['ci95_hi_pct'] - bench_compare['our_success_rate_pct']).abs()
)

display(Markdown('### Comparison with RoboCasa GR1 posted benchmark rates'))
display(bench_compare.sort_values('scenario_id').reset_index(drop=True))

plot_df = bench_compare[bench_compare['benchmark_success_rate_pct'].notna()].copy()
if not plot_df.empty:
    x = np.arange(len(plot_df))
    w = 0.28
    fig, ax = plt.subplots(figsize=(11.4, 5.4))
    ax.bar(x - w, plot_df['our_success_rate_pct'], width=w, label='our sim success (episodes 2-21)')
    ax.bar(x, plot_df['benchmark_success_rate_pct'], width=w, label='posted benchmark')
    ax.bar(x + w, plot_df['difference_sim_minus_benchmark_pct'], width=w, label='difference (sim - benchmark)')
    ax.errorbar(
        x - w,
        plot_df['our_success_rate_pct'],
        yerr=[plot_df['our_success_rate_pct'] - plot_df['ci95_lo_pct'], plot_df['ci95_hi_pct'] - plot_df['our_success_rate_pct']],
        fmt='none',
        ecolor='black',
        capsize=4,
        linewidth=1.1,
    )
    ax.axhline(0.0, color='gray', linewidth=1.0, linestyle='--')
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df['scenario_label'], rotation=20, ha='right')
    ax.set_ylabel('rate (%)')
    ax.set_xlabel('scenario')
    ax.set_title('Our observed success vs posted GR1 benchmark (+ difference column)')
    ax.legend(loc='best')
    plt.tight_layout()
    plt.show()

if not plot_df.empty:
    better = int((plot_df['difference_sim_minus_benchmark_pct'] > 0).sum())
    worse = int((plot_df['difference_sim_minus_benchmark_pct'] < 0).sum())
    within_ci = int(plot_df['benchmark_within_our_ci95'].sum())
    display(Markdown(
        '### Section 1 Response: Benchmark comparison interpretation\n'
        f'- Matched scenarios with posted benchmark entries: **{len(plot_df)}**.\n'
        f'- Our observed success is higher on **{better}** scenario(s) and lower on **{worse}** scenario(s).\n'
        f'- Benchmark rate falls inside our 95% confidence interval for **{within_ci}/{len(plot_df)}** matched scenarios.\n'
        '- Interpretation note: this is a protocol-level comparison proxy; differences may include policy/evaluation setup effects beyond random sampling error.'
    ))

### Overview Interpretation Notes

Use the tables and plots above to identify:
- Cases where simulator success is high but BT completion proxy is lower (interpretation mismatch risk).
- Cases where BT completion appears high while simulator success is lower (possible simulator-side failures, state tracking issues, or BT overconfidence).
- Model-specific scenario sensitivity based on gap magnitude and agreement profile.

## Section 2: RQ1 - BT Structure and Interpretability

**RQ1:** Can generated BTs represent scenario tasks as interpretable subgoal structures for success characterization?

This section addresses RQ1a-RQ1e with dedicated cells for each subquestion.

### RQ1a: Structural validity, parseability, and non-empty BT behavior

Proxy used in this analysis: per-episode BT completion evidence (`model_status_norm == complete`) compared against simulator success.

In [ ]:
RQ1_TABLES = build_rq1_tables(COMPARE_DF)
display(RQ1_TABLES['rq1a_structure_success'].sort_values(['scenario_id', 'model_name']).reset_index(drop=True))
plot_rq1_bundle(RQ1_TABLES)

### RQ1b: Task-relevant nodes and success criteria presence

Using surveyed node rows as a practical proxy for BT decomposition richness:
- Unique `status_eval_id` coverage per scenario/model.
- Fraction of rows with complete/running/failed status evidence.

In [ ]:
rq1b_nodes = (
    COMPARE_DF.groupby(['model_name', 'scenario_id'], dropna=False)
    .agg(
        rows=('model_name', 'size'),
        unique_status_eval_ids=('status_eval_id', lambda s: int(pd.Series(s).astype(str).nunique())),
        complete_rows=('model_status_norm', lambda s: int((pd.Series(s) == 'complete').sum())),
        running_rows=('model_status_norm', lambda s: int((pd.Series(s) == 'running').sum())),
        failed_rows=('model_status_norm', lambda s: int((pd.Series(s) == 'failed').sum())),
    )
    .reset_index()
)

rq1b_nodes['complete_rate_pct'] = (100.0 * rq1b_nodes['complete_rows'] / rq1b_nodes['rows'].replace(0, np.nan)).round(1)
rq1b_nodes['running_rate_pct'] = (100.0 * rq1b_nodes['running_rows'] / rq1b_nodes['rows'].replace(0, np.nan)).round(1)
rq1b_nodes['failed_rate_pct'] = (100.0 * rq1b_nodes['failed_rows'] / rq1b_nodes['rows'].replace(0, np.nan)).round(1)
rq1b_nodes['scenario_label'] = rq1b_nodes['scenario_id'].map(scenario_label)

display(rq1b_nodes.sort_values(['scenario_id', 'model_name']).reset_index(drop=True))

pivot_nodes = rq1b_nodes.pivot(index='scenario_label', columns='model_name', values='unique_status_eval_ids').fillna(0.0)
ax = pivot_nodes.plot(kind='bar', figsize=(9.0, 4.8), rot=20)
ax.set_title('RQ1b proxy: unique status_eval_id coverage by scenario')
ax.set_xlabel('scenario')
ax.set_ylabel('unique node ids observed')
plt.tight_layout()
plt.show()

### RQ1c: Human interpretability and usefulness judgments

Human agreement is taken from reviewer labels: `y`, `p`, `n`, plus timing-only alignment (`timing_aligned`) when reviewer status is blank but timing matches.

In [ ]:
rq1c = RQ1_TABLES['rq1b_interpretability_agreement'].copy()
display(rq1c.sort_values(['model_name', 'agreement_effective']).reset_index(drop=True))

if not rq1c.empty:
    p = rq1c.pivot(index='model_name', columns='agreement_effective', values='rate_pct').fillna(0.0)
    order = [c for c in ['y', 'p', 'timing_aligned', 'n'] if c in p.columns]
    p = p[order]
    ax = p.plot(kind='bar', stacked=True, figsize=(8.8, 4.8), rot=0)
    ax.set_title('RQ1c: Human interpretability profile by mission engine')
    ax.set_xlabel('model')
    ax.set_ylabel('rate (%)')
    plt.tight_layout()
    plt.show()

### RQ1d: Scenario reuse across repeated seeds/initializations

We estimate reuse consistency from scenario-level variability in sim-minus-BT gap: lower variance indicates more stable reuse behavior across episodes/runs.

In [ ]:
rq1d_reuse = RQ1_TABLES['rq1e_reuse_consistency'].copy()
display(rq1d_reuse)

if not rq1d_reuse.empty:
    ax = rq1d_reuse.set_index('model_name')['std_sim_minus_bt_gap_pct'].plot(kind='bar', figsize=(7.0, 4.2), rot=0)
    ax.set_title('RQ1d proxy: reuse variability (std of sim-BT gap)')
    ax.set_xlabel('model')
    ax.set_ylabel('std gap (%)')
    plt.tight_layout()
    plt.show()

### RQ1e: BT decomposition error taxonomy and audit impact

Disagreement (`n`) rows are grouped by normalized model status to expose dominant decomposition/audit error modes.

In [ ]:
rq1e = RQ1_TABLES['rq1d_error_taxonomy'].copy()
display(rq1e)

if not rq1e.empty:
    for m in sorted(rq1e['model_name'].dropna().unique()):
        g = rq1e[rq1e['model_name'] == m].sort_values('count', ascending=False).head(10)
        ax = g.set_index('model_status_norm')['count'].plot(kind='bar', figsize=(8.0, 4.2), rot=20)
        ax.set_title(f'RQ1e disagreement taxonomy ({m})')
        ax.set_xlabel('model_status_norm')
        ax.set_ylabel('count')
        plt.tight_layout()
        plt.show()

In [ ]:
rq1a_tbl = RQ1_TABLES['rq1a_structure_success'].copy()
rq1b_tbl = RQ1_TABLES['rq1b_interpretability_agreement'].copy()
rq1d_tbl = RQ1_TABLES['rq1d_error_taxonomy'].copy()
rq1e_tbl = RQ1_TABLES['rq1e_reuse_consistency'].copy()

rq1a_msg = '- RQ1a: No structure-success rows available.'
if not rq1a_tbl.empty:
    mean_abs_gap = float(rq1a_tbl['sim_minus_bt_gap_pct'].abs().mean())
    rq1a_msg = f'- RQ1a: Across scenarios/models, the average absolute sim-vs-BT gap is **{mean_abs_gap:.1f} pct-points**, indicating the BT-complete proxy is informative but not equivalent to simulator success.'

rq1b_msg = '- RQ1b: No interpretability-agreement rows available.'
if not rq1b_tbl.empty:
    ypt = rq1b_tbl[rq1b_tbl['agreement_effective'].isin(['y', 'p', 'timing_aligned'])].groupby('model_name', dropna=False)['rate_pct'].sum()
    if len(ypt) > 0:
        best_model = ypt.idxmax()
        best_rate = ypt.max()
        rq1b_msg = f'- RQ1b: The highest reviewer-positive composition (`y`/`p`/timing-aligned) is **{best_model}** at **{best_rate:.1f}%**, supporting practical BT interpretability in the surveyed subset.'

rq1c_msg = '- RQ1c: Reviewer judgment summary unavailable.'
if not rq1b_tbl.empty:
    n_rate = rq1b_tbl[rq1b_tbl['agreement_effective'] == 'n'].groupby('model_name', dropna=False)['rate_pct'].sum()
    if len(n_rate) > 0:
        worst_model = n_rate.idxmax()
        worst_rate = n_rate.max()
        rq1c_msg = f'- RQ1c: Disagreement (`n`) remains concentrated in **{worst_model}** at **{worst_rate:.1f}%**, highlighting where interpretability remains fragile.'

rq1d_msg = '- RQ1d: Reuse-consistency table unavailable.'
if not rq1e_tbl.empty:
    stable_idx = rq1e_tbl['std_sim_minus_bt_gap_pct'].fillna(np.inf).idxmin()
    stable_row = rq1e_tbl.loc[stable_idx]
    rq1d_msg = f"- RQ1d: Lower cross-scenario variability in sim-minus-BT gap for **{stable_row['model_name']}** (std **{stable_row['std_sim_minus_bt_gap_pct']:.2f}**) suggests better reuse consistency under repeated seeds/initializations."

rq1e_msg = '- RQ1e: Error taxonomy rows unavailable.'
if not rq1d_tbl.empty:
    top_err = rq1d_tbl.sort_values('count', ascending=False).iloc[0]
    rq1e_msg = f"- RQ1e: The dominant disagreement mode is **{top_err['model_status_norm']}** for **{top_err['model_name']}** (count **{int(top_err['count'])}**), indicating where BT decomposition or status interpretation most often breaks audit alignment."

display(Markdown(
    '### RQ1 Responses (Data-Backed)\n'
    + rq1a_msg + '\n'
    + rq1b_msg + '\n'
    + rq1c_msg + '\n'
    + rq1d_msg + '\n'
    + rq1e_msg
))

## Section 3: RQ2 - Node Status, Progress, and Failure Signaling

**RQ2:** Are VLM node-status and progress scores beneficial, and do they coincide with human progress and failure judgments?

This section addresses RQ2a-RQ2e with dedicated tables/plots and synthesis notes.

In [ ]:
RQ2_TABLES = build_rq2_tables(COMPARE_DF)
plot_rq2_bundle(RQ2_TABLES, COMPARE_DF)

### RQ2a: Progress-score correlation with human progress judgment

In [ ]:
rq2a = RQ2_TABLES['rq2a_progress_alignment'].copy()
display(rq2a)

prog_df = COMPARE_DF[pd.to_numeric(COMPARE_DF['model_progress'], errors='coerce').notna()].copy()
if not prog_df.empty:
    prog_df['reviewer_positive'] = prog_df['agreement_effective'].isin(['y', 'p', 'timing_aligned']).astype(int)
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6), sharey=True)
    models = sorted(prog_df['model_name'].dropna().unique())
    for i, m in enumerate(models[:2]):
        g = prog_df[prog_df['model_name'] == m]
        axes[i].scatter(g['model_progress'], g['reviewer_positive'], alpha=0.18, s=10)
        axes[i].set_title(f'{m}: progress vs reviewer-positive')
        axes[i].set_xlabel('model_progress')
    axes[0].set_ylabel('reviewer_positive (0/1)')
    plt.tight_layout()
    plt.show()

### RQ2b: Node-status agreement with human labels

In [ ]:
rq2b = RQ2_TABLES['rq2b_status_agreement'].copy()
display(rq2b)

if not rq2b.empty:
    x = np.arange(len(rq2b))
    w = 0.36
    fig, ax = plt.subplots(figsize=(8.2, 4.6))
    ax.bar(x - w / 2, rq2b['effective_agreement_rate_pct'], width=w, label='effective agreement')
    ax.bar(x + w / 2, rq2b['strict_yes_rate_pct'], width=w, label='strict yes')
    ax.set_xticks(x)
    ax.set_xticklabels(rq2b['model_name'])
    ax.set_ylabel('rate (%)')
    ax.set_xlabel('model')
    ax.set_title('RQ2b: node-status agreement rates')
    ax.legend(loc='best')
    plt.tight_layout()
    plt.show()

### RQ2c: Early stagnation/failure signaling before terminal outcome

In [ ]:
rq2c = RQ2_TABLES['rq2c_early_failure'].copy()
display(rq2c)

if not rq2c.empty:
    fig, ax = plt.subplots(figsize=(8.0, 4.6))
    ax.bar(rq2c['model_name'], rq2c['timing_within_2s_rate_pct'])
    ax.set_title('RQ2c: timing within +/-2s (higher is better alignment)')
    ax.set_xlabel('model')
    ax.set_ylabel('rate (%)')
    plt.tight_layout()
    plt.show()

failed_disagree = COMPARE_DF[(COMPARE_DF['model_status_norm'] == 'failed') & (COMPARE_DF['agreement_effective'] == 'n')]
display(Markdown(f'Rows with failed status + reviewer disagreement: **{len(failed_disagree)}**'))

### RQ2d: Disagreement taxonomy and diagnostic usefulness

In [ ]:
rq2d = RQ2_TABLES['rq2d_disagreement_taxonomy'].copy()
display(rq2d)

if not rq2d.empty:
    for m in sorted(rq2d['model_name'].dropna().unique()):
        g = rq2d[rq2d['model_name'] == m].sort_values('count', ascending=False).head(10)
        ax = g.set_index('model_status_norm')['count'].plot(kind='bar', figsize=(8.2, 4.4), rot=20)
        ax.set_title(f'RQ2d disagreement types ({m})')
        ax.set_xlabel('model_status_norm')
        ax.set_ylabel('count')
        plt.tight_layout()
        plt.show()

disagree_rows = COMPARE_DF[COMPARE_DF['agreement_effective'] == 'n'].copy()
disagree_rows['reviewer_status_clean'] = disagree_rows['reviewer_status_clean'].fillna('')
disagree_rows['disagreement_bucket'] = disagree_rows.apply(disagreement_bucket, axis=1)

bucket_table = (
    disagree_rows.groupby(['model_name', 'model_status_norm', 'disagreement_bucket'], dropna=False)
    .size()
    .reset_index(name='count')
)
bucket_table['rate_within_model_status_pct'] = (
    100.0 * bucket_table['count'] / bucket_table.groupby(['model_name', 'model_status_norm'])['count'].transform('sum').replace(0, np.nan)
).round(1)

display(Markdown('### RQ2d deep taxonomy by disagreement bucket'))
display(bucket_table.sort_values(['model_name', 'model_status_norm', 'count'], ascending=[True, True, False]).reset_index(drop=True))

for m in sorted(bucket_table['model_name'].dropna().unique()):
    g = bucket_table[bucket_table['model_name'] == m].copy()
    p = g.pivot(index='model_status_norm', columns='disagreement_bucket', values='rate_within_model_status_pct').fillna(0.0)
    if p.empty:
        continue
    ax = p.plot(kind='bar', stacked=True, figsize=(11.0, 4.8), rot=20)
    ax.set_title(f'RQ2d deep view ({m}): disagreement bucket mix within model_status')
    ax.set_xlabel('model_status_norm')
    ax.set_ylabel('rate within model_status (%)')
    ax.legend(title='bucket', bbox_to_anchor=(1.01, 1.0), loc='upper left')
    plt.tight_layout()
    plt.show()

running_qwen3 = disagree_rows[
    (disagree_rows['model_name'] == 'qwen3')
    & (disagree_rows['model_status_norm'] == 'running')
]
if not running_qwen3.empty:
    q3_run_bucket = (
        running_qwen3.groupby('disagreement_bucket', dropna=False)
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .reset_index(drop=True)
    )
    q3_run_bucket['rate_pct'] = (100.0 * q3_run_bucket['count'] / q3_run_bucket['count'].sum()).round(1)
    display(Markdown('### RQ2d focused drill-down: qwen3 running disagreements'))
    display(q3_run_bucket)

    ax = q3_run_bucket.set_index('disagreement_bucket')['count'].plot(kind='bar', figsize=(9.4, 4.6), rot=20)
    ax.set_title('qwen3 running disagreements by subtype')
    ax.set_xlabel('disagreement subtype')
    ax.set_ylabel('count')
    plt.tight_layout()
    plt.show()

    status_notes = running_qwen3[['reviewer_status_clean', 'reviewer_notes']].copy()
    status_notes['snippet'] = (
        status_notes['reviewer_status_clean'].fillna('').astype(str).str.strip()
        + ' | '
        + status_notes['reviewer_notes'].fillna('').astype(str).str.strip()
    ).str.strip(' |')
    snippet_top = (
        status_notes[status_notes['snippet'].ne('')]
        .groupby('snippet', dropna=False)
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .head(15)
        .reset_index(drop=True)
    )
    display(Markdown('Top reviewer evidence snippets for qwen3 running disagreements'))
    display(snippet_top)

    time_only_cnt = int(q3_run_bucket.loc[q3_run_bucket['disagreement_bucket'].str.contains('time_only', na=False), 'count'].sum())
    status_override_cnt = int(q3_run_bucket['count'].sum() - time_only_cnt)
    display(Markdown(
        '### RQ2d Response\n'
        '- For qwen3 rows labeled `running` with reviewer disagreement, decomposition shows whether disagreement is primarily timing-only or semantic status/node override.\n'
        f'- Timing-only disagreements: **{time_only_cnt}**; status-override disagreements: **{status_override_cnt}**.\n'
        '- The bucket and snippet tables identify where disagreement is likely due to node-transition mismatch versus failure/progress/completion interpretation mismatch.'
    ))
else:
    display(Markdown('### RQ2d Response\nNo qwen3 running disagreement rows were found in the filtered dataset.'))

### RQ2e: Added value of audit outputs beyond simulator success/failure

This comparison highlights where simulator success alone can miss explainable BT/VLM-human disagreements.

In [ ]:
episode_level = (
    COMPARE_DF.groupby(['model_name', 'episode_dir', 'scenario_id'], dropna=False)
    .agg(
        sim_success=('sim_success', 'first'),
        bt_complete=('model_status_norm', lambda s: bool((pd.Series(s) == 'complete').any())),
        any_disagreement=('agreement_effective', lambda s: bool((pd.Series(s) == 'n').any())),
    )
    .reset_index()
)

rq2e = (
    episode_level.groupby('model_name', dropna=False)
    .agg(
        episodes=('episode_dir', 'nunique'),
        sim_success_rate_pct=('sim_success', lambda s: round(100.0 * pd.Series(s).eq(True).mean(), 1) if len(pd.Series(s).dropna()) else np.nan),
        bt_complete_rate_pct=('bt_complete', lambda s: round(100.0 * pd.Series(s).eq(True).mean(), 1)),
        any_disagreement_rate_pct=('any_disagreement', lambda s: round(100.0 * pd.Series(s).eq(True).mean(), 1)),
    )
    .reset_index()
)

display(rq2e)

if not rq2e.empty:
    x = np.arange(len(rq2e))
    w = 0.25
    fig, ax = plt.subplots(figsize=(9.0, 4.8))
    ax.bar(x - w, rq2e['sim_success_rate_pct'], width=w, label='sim success')
    ax.bar(x, rq2e['bt_complete_rate_pct'], width=w, label='BT complete')
    ax.bar(x + w, rq2e['any_disagreement_rate_pct'], width=w, label='any disagreement')
    ax.set_xticks(x)
    ax.set_xticklabels(rq2e['model_name'])
    ax.set_xlabel('model')
    ax.set_ylabel('rate (%)')
    ax.set_title('RQ2e: audit utility beyond simulator-only outcomes')
    ax.legend(loc='best')
    plt.tight_layout()
    plt.show()

In [ ]:
rq2a_tbl = RQ2_TABLES['rq2a_progress_alignment'].copy()
rq2b_tbl = RQ2_TABLES['rq2b_status_agreement'].copy()
rq2c_tbl = RQ2_TABLES['rq2c_early_failure'].copy()
rq2d_tbl = RQ2_TABLES['rq2d_disagreement_taxonomy'].copy()

rq2a_msg = '- RQ2a: No progress-alignment rows available.'
if not rq2a_tbl.empty and rq2a_tbl['spearman_progress_vs_reviewer_positive'].notna().any():
    top = rq2a_tbl.sort_values('spearman_progress_vs_reviewer_positive', ascending=False).iloc[0]
    rq2a_msg = f"- RQ2a: Highest progress-vs-reviewer-positive Spearman correlation is **{top['model_name']}** at **{top['spearman_progress_vs_reviewer_positive']:.3f}**, supporting partial usefulness of model progress scores."

rq2b_msg = '- RQ2b: No status-agreement rows available.'
if not rq2b_tbl.empty:
    top2 = rq2b_tbl.sort_values('effective_agreement_rate_pct', ascending=False).iloc[0]
    rq2b_msg = f"- RQ2b: Best effective node-status agreement is **{top2['model_name']}** at **{top2['effective_agreement_rate_pct']:.1f}%** (strict-yes **{top2['strict_yes_rate_pct']:.1f}%**)."

rq2c_msg = '- RQ2c: No timing alignment rows available.'
if not rq2c_tbl.empty:
    top3 = rq2c_tbl.sort_values('timing_within_2s_rate_pct', ascending=False).iloc[0]
    rq2c_msg = f"- RQ2c: Earliest alignment proxy (timing within +/-2s) is strongest for **{top3['model_name']}** at **{top3['timing_within_2s_rate_pct']:.1f}%**."

rq2d_msg = '- RQ2d: No disagreement taxonomy rows available.'
if not rq2d_tbl.empty:
    top4 = rq2d_tbl.sort_values('count', ascending=False).iloc[0]
    rq2d_msg = f"- RQ2d: Most common disagreement type is **{top4['model_status_norm']}** for **{top4['model_name']}** (count **{int(top4['count'])}**); see deep subtype drill-down above for diagnostic categories."

rq2e_msg = '- RQ2e: No episode-level audit utility table available.'
if 'rq2e' in globals() and isinstance(rq2e, pd.DataFrame) and (not rq2e.empty):
    top5 = rq2e.sort_values('any_disagreement_rate_pct', ascending=False).iloc[0]
    rq2e_msg = f"- RQ2e: **{top5['model_name']}** shows **{top5['any_disagreement_rate_pct']:.1f}%** episode-level disagreement presence, indicating additional post-hoc diagnostic signal beyond simulator success/failure alone."

display(Markdown(
    '### RQ2 Responses (Data-Backed)\n'
    + rq2a_msg + '\n'
    + rq2b_msg + '\n'
    + rq2c_msg + '\n'
    + rq2d_msg + '\n'
    + rq2e_msg
))

In [ ]:
# Episode-level decomposition table: Sim+BT / Sim only / BT only / Neither
import numpy as np
import pandas as pd
from IPython.display import display, Markdown


def _first_existing(df: pd.DataFrame, candidates: list[str]) -> str | None:
    cols = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in cols:
            return cols[c.lower()]
    return None


def _normalize_model_name(v: str) -> str:
    s = str(v).strip().lower()
    mapping = {
        "qwen3": "Qwen3",
        "qwen3.5": "Qwen3.5",
        "qwen3.5_tailored": "Qwen3.5 tailored",
        "qwen3_5": "Qwen3.5",
        "qwen3_5_tailored": "Qwen3.5 tailored",
    }
    return mapping.get(s, str(v))


def _clean_scenario_name(v: str) -> str:
    s = str(v)
    s = s.replace("gr1_unified/", "")
    s = s.replace("_GR1ArmsAndWaistFourierHands_Env", "")
    mapping = {
        "PnPCanToDrawerClose": "Can to Drawer Close",
        "PosttrainPnPNovelFromCuttingboardToBasketSplitA": "Cuttingboard to Basket",
        "PosttrainPnPNovelFromPlateToPlateSplitA": "Plate to Plate",
    }
    return mapping.get(s, s)


def _decomp_label(sim_success: bool, bt_complete: bool) -> str:
    if sim_success and bt_complete:
        return "Sim + BT"
    if sim_success and (not bt_complete):
        return "Sim only"
    if (not sim_success) and bt_complete:
        return "BT only"
    return "Neither"


# Build episode-level decomposition from EPISODE_MODEL_DF if available.
source_df = None
if "EPISODE_MODEL_DF" in globals() and isinstance(EPISODE_MODEL_DF, pd.DataFrame) and not EPISODE_MODEL_DF.empty:
    source_df = EPISODE_MODEL_DF.copy()
elif "COMPARE_DF" in globals() and isinstance(COMPARE_DF, pd.DataFrame) and not COMPARE_DF.empty:
    source_df = COMPARE_DF.copy()

if source_df is None or source_df.empty:
    raise ValueError("No suitable source dataframe found. Expected EPISODE_MODEL_DF or COMPARE_DF.")

col_model = _first_existing(source_df, ["model_name", "model", "mission_engine", "engine"])
col_scenario = _first_existing(source_df, ["scenario_display", "scenario", "scenario_id", "condition"])
col_episode = _first_existing(source_df, ["episode_dir", "episode_id", "episode"])
col_sim = _first_existing(source_df, ["sim_success", "success", "sim_success_int", "success_int"])
col_bt = _first_existing(source_df, ["bt_complete_proxy", "bt_complete", "bt_success", "complete"])

if not all([col_model, col_scenario, col_episode, col_sim]):
    missing = {
        "model": col_model,
        "scenario": col_scenario,
        "episode": col_episode,
        "sim_success": col_sim,
    }
    raise ValueError(f"Missing required columns in source dataframe: {missing}")

work = source_df.copy()

# If BT-complete flag is missing, infer from model-status signals when possible.
if col_bt is None:
    col_status = _first_existing(work, ["model_status_norm", "model_status", "status"])
    if col_status is None:
        raise ValueError("Could not locate BT completion indicator or status column to infer it.")
    work["__bt_complete_inferred"] = work[col_status].astype(str).str.strip().str.lower().eq("complete")
    col_bt = "__bt_complete_inferred"

# Collapse to one row per (model, scenario, episode).
episode_level = (
    work.groupby([col_model, col_scenario, col_episode], dropna=False)
    .agg(
        sim_success=(col_sim, lambda s: bool(pd.Series(s).astype(bool).any())),
        bt_complete=(col_bt, lambda s: bool(pd.Series(s).astype(bool).any())),
    )
    .reset_index()
)

episode_level["Model"] = episode_level[col_model].map(_normalize_model_name)
episode_level["Scenario"] = episode_level[col_scenario].map(_clean_scenario_name)
episode_level["bucket"] = [
    _decomp_label(sim_ok, bt_ok)
    for sim_ok, bt_ok in zip(episode_level["sim_success"], episode_level["bt_complete"])
]

# Keep only manuscript models by default.
model_order = ["Qwen3", "Qwen3.5", "Qwen3.5 tailored"]
episode_level = episode_level[episode_level["Model"].isin(model_order)].copy()

order_cols = ["Sim + BT", "Sim only", "BT only", "Neither"]

table_counts = (
    episode_level.groupby(["Model", "Scenario", "bucket"], dropna=False)
    .size()
    .reset_index(name="count")
    .pivot_table(index=["Model", "Scenario"], columns="bucket", values="count", fill_value=0)
    .reindex(columns=order_cols, fill_value=0)
    .reset_index()
)

totals = table_counts[order_cols].sum(axis=1).replace(0, np.nan)
table_pct = table_counts.copy()
for c in order_cols:
    table_pct[c] = (100.0 * table_counts[c] / totals).round(1)

table_counts["_model_order"] = table_counts["Model"].apply(model_order.index)
table_counts = table_counts.sort_values(["_model_order", "Scenario"]).drop(columns=["_model_order"]).reset_index(drop=True)

table_pct = table_pct.set_index(["Model", "Scenario"]).loc[table_counts.set_index(["Model", "Scenario"]).index].reset_index()

# Store for downstream cells/use.
SIM_BT_DECOMP_TABLE_COUNTS = table_counts
SIM_BT_DECOMP_TABLE_PCT = table_pct

# Display tables.
display(Markdown("### Episode-Level Decomposition Table (Counts)"))
display(SIM_BT_DECOMP_TABLE_COUNTS)

display(Markdown("### Episode-Level Decomposition Table (Percent, %)"))
display(SIM_BT_DECOMP_TABLE_PCT)

# Build LaTeX with percentages (counts shown in parentheses).
latex_rows = []
for _, r in table_counts.iterrows():
    rp = table_pct[(table_pct["Model"] == r["Model"]) & (table_pct["Scenario"] == r["Scenario"])].iloc[0]

    def _fmt(col: str) -> str:
        return f"{rp[col]:.1f}\\% ({int(r[col])})"

    latex_rows.append(
        f"{r['Model']} & {r['Scenario']} & {_fmt('Sim + BT')} & {_fmt('Sim only')} & {_fmt('BT only')} & {_fmt('Neither')} \\\\"
    )

latex_table = "\n".join([
    "\\begin{table*}[t]",
    "\\centering",
    "\\caption{Episode-level decomposition of simulator success and BT completion.}",
    "\\label{tab:sim_bt_decomposition}",
    "\\begin{tabular}{p{0.18\\textwidth} p{0.22\\textwidth} p{0.12\\textwidth} p{0.12\\textwidth} p{0.12\\textwidth} p{0.12\\textwidth}}",
    "\\toprule",
    "\\textbf{Model} & \\textbf{Scenario} & \\textbf{Sim + BT} & \\textbf{Sim only} & \\textbf{BT only} & \\textbf{Neither} \\\\",
    "\\midrule",
    *latex_rows,
    "\\bottomrule",
    "\\end{tabular}",
    "\\end{table*}",
])

SIM_BT_DECOMP_LATEX = latex_table

display(Markdown("### LaTeX Snippet"))
print(SIM_BT_DECOMP_LATEX)

## Summary Guidance and Reporting Checklist

Use this notebook output to write final results with explicit claims for each research question and subquestion.

Include the following in your written results section:
- Section 1 decomposition findings: `sim_and_bt`, `sim_only`, `bt_only`, `neither` rates by scenario and mission engine.
- Benchmark comparison findings: matched-scenario deltas versus posted GR1 rates, confidence intervals, and whether differences are plausibly within sampling error.
- RQ1 responses: use the generated `RQ1 Responses (Data-Backed)` bullets and cite supporting tables/plots.
- RQ2 responses: use the generated `RQ2 Responses (Data-Backed)` bullets and cite supporting tables/plots.
- RQ2d deep diagnostics: for each model-status disagreement type, report timing-only versus status-override proportions and representative reviewer evidence snippets.

Interpretation caveat for academic reporting: simulator success, BT completion proxy, and reviewer agreement measure different constructs. Agreement among all three is strongest evidence; mismatch patterns should be interpreted as diagnostic signals rather than direct contradictions.